In [ ]:
from neural_lam import metrics

# Example usage cell (paste in the notebook)
# root = "/mimer/NOBACKUP/groups/mlhighres/users/mikhaili/neural-lam/output/230126"
output_dir="output_test_r1_final"
root = "/mimer/NOBACKUP/groups/mlhighres/users/mikhaili/neural-lam/output/270126"
experiments = ["test_SI_small_r1", "test_CorrDiff_small_r1", "test_UNet_small_r1"]
variables = ["pr", "tas"]
start_date = "2010-01-01"
end_date = "2014-12-31"

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from typing import Optional, Tuple, Dict


def _rmse_array(a: np.ndarray, b: np.ndarray) -> float:
    m = np.isfinite(a) & np.isfinite(b)
    if not m.any():
        return float("nan")
    diff = a[m].ravel() - b[m].ravel()
    return float(np.sqrt(np.mean(diff ** 2)))

def daily_and_overall_rmse_from_netcdf(
    ensemble_mean_nc: str,
    target_nc: str,
    var_name: Optional[str] = None,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    save_csv: Optional[str] = None,
    plot: bool = True
) -> Tuple[pd.Series, float]:
    """
    Compute daily RMSE between ensemble mean and target from two netCDF files.
    - ensemble_mean_nc, target_nc: file paths
    - var_name: variable name inside files; if None picks the first data variable
    - start_date/end_date: 'YYYY-MM-DD' to restrict range (inclusive)
    Returns (daily_rmse_series indexed by date ISO string, overall_rmse float).
    """
    ds_mean = xr.open_dataset(ensemble_mean_nc)
    ds_tgt = xr.open_dataset(target_nc)

    # pick variable if not provided
    if var_name is None:
        # choose first data variable not named 'time' / coords
        data_vars = [v for v in ds_mean.data_vars.keys()]
        if not data_vars:
            raise RuntimeError("No data variables found in ensemble mean file")
        var_name = data_vars[0]

    # optionally restrict time range
    sel_slice = slice(start_date, end_date) if start_date or end_date else slice(None)
    ds_mean = ds_mean.sel(time=sel_slice)
    ds_tgt = ds_tgt.sel(time=sel_slice)

    # intersect available times
    times_mean = np.asarray(ds_mean["time"].values)
    times_tgt = np.asarray(ds_tgt["time"].values)
    common_times = np.intersect1d(times_mean, times_tgt)
    if common_times.size == 0:
        raise RuntimeError("No overlapping times between ensemble mean and target datasets")

    # select only common times (keeps original order present in mean)
    ds_mean = ds_mean.sel(time=common_times)
    ds_tgt = ds_tgt.sel(time=common_times)

    daily = {}
    # compute RMSE per day
    for t in ds_mean.time.values:
        a = ds_mean[var_name].sel(time=t).values
        b = ds_tgt[var_name].sel(time=t).values
        r = _rmse_array(a, b)
        date_str = np.datetime_as_string(t, unit='D')
        daily[date_str] = r

    daily_series = pd.Series(daily).sort_index()

    # overall RMSE across all times and spatial points (ignores NaNs)
    a_all = ds_mean[var_name].values
    b_all = ds_tgt[var_name].values
    overall = _rmse_array(a_all, b_all)

    if save_csv:
        df = pd.DataFrame({"date": daily_series.index, "rmse": daily_series.values})
        df.to_csv(save_csv, index=False)
        print(f"Saved daily RMSE to {save_csv}")

    if plot:
        dates = pd.to_datetime(daily_series.index)
        fig, ax = plt.subplots(figsize=(10,4))
        ax.plot(dates, daily_series.values, marker='o', linestyle='-')
        ax.set_xlabel("Date")
        ax.set_ylabel("Daily RMSE")
        ax.set_title(f"Daily RMSE (ensemble mean) {common_times[0].astype('M8[D]').astype(str)} to {common_times[-1].astype('M8[D]').astype(str)}\nOverall RMSE={overall:.4f}")
        ax.grid(True, ls=':', alpha=0.6)
        fig.autofmt_xdate()
        plt.show()

    ds_mean.close()
    ds_tgt.close()
    return daily_series, overall



In [ ]:
import os
import glob
import pandas as pd
from typing import List, Tuple, Dict, Optional

def strip_prefix_up_to_underscore(name: str) -> str:
    """Return part after first underscore, or original if no underscore."""
    return name.split("_", 1)[1] if "_" in name else name

def _find_nc_file(root: str, var: str, tag: str, exp: str) -> Optional[str]:
    # tag: "ensemble_mean" or "target" (or "ensemble_std", "ensemble_member_0", ...)
    # Try several filename patterns from most-specific to most-general so files like
    # "tas_target_SI_2010-01-01_2014-12-31.nc" are found even when exp=="test_SI_small_r1".
    exp_suffix = strip_prefix_up_to_underscore(exp)
    parts = exp.split("_")
    pats = []
    # prefer files inside experiment directory
    exp_dir = os.path.join(root, exp)
    pats.append(os.path.join(exp_dir, f"{var}*{tag}*{exp_suffix}*.nc"))
    pats.append(os.path.join(exp_dir, f"{var}*{tag}*{exp}*.nc"))
    # try second token (e.g. "SI" from "test_SI_small_r1")
    if len(parts) > 1:
        pats.append(os.path.join(exp_dir, f"{var}*{tag}*{parts[1]}*.nc"))
    # more relaxed patterns
    pats.append(os.path.join(exp_dir, f"{var}*{tag}*.nc"))
    pats.append(os.path.join(exp_dir, f"*{tag}*.nc"))
    # fall back to searching root/exp recursively (in case files are elsewhere)
    for pat in pats:
        matches = sorted(glob.glob(pat))
        if matches:
            return matches[0]
    # try a looser search across root/exp subdirs
    loose_pat = os.path.join(root, exp, "**", f"*{tag}*.nc")
    matches = sorted(glob.glob(loose_pat, recursive=True))
    if matches:
        return matches[0]
    return None

def multi_model_rmse_plot(
    root_dir: str,
    experiments: List[str],
    variables: List[str],
    start_date: str,
    end_date: str,
    save_csv_dir: Optional[str] = None,
    plot: bool = True,
) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame]:
    """
    For each variable, compute daily RMSE series for each experiment and plot them
    on the same figure. Returns:
      - dict mapping variable -> DataFrame (index dates, columns experiments)
      - overall_summary DataFrame (experiment x variable overall RMSE)
    """
    all_daily = {}
    overall_rows = []
    for var in variables:
        print(f"Processing variable: {var}")
        daily_dfs = []
        for exp in experiments:
            print(f"  Processing experiment: {exp}")
            mean_nc = _find_nc_file(root_dir, var, "ensemble_mean", exp)
            target_nc = _find_nc_file(root_dir, var, "target", exp)
            if mean_nc is None or target_nc is None:
                print(f"Skipping {var} / {exp}: missing files (mean={bool(mean_nc)}, target={bool(target_nc)})")
                continue
            # compute daily series without plotting
            daily_series, overall = daily_and_overall_rmse_from_netcdf(
                mean_nc, target_nc, var_name=var, start_date=start_date, end_date=end_date, save_csv=None, plot=False
            )
            s = daily_series.rename(exp)
            daily_dfs.append(s)
            overall_rows.append({"experiment": exp, "variable": var, "overall_rmse": overall})
        if not daily_dfs:
            print(f"No data for variable {var}")
            continue
        # align by index (dates); outer join to keep full date range
        df_var = pd.concat(daily_dfs, axis=1)
        df_var = df_var.sort_index()
        all_daily[var] = df_var

        # plot if requested
        if plot:
            fig, ax = plt.subplots(figsize=(10, 4))
            for col in df_var.columns:
                ax.plot(pd.to_datetime(df_var.index), df_var[col], marker='o', linestyle='-', label=col)
            ax.set_title(f"Daily RMSE (ensemble mean) for variable '{var}'\n{start_date} to {end_date}")
            ax.set_xlabel("Date")
            ax.set_ylabel("Daily RMSE")
            ax.grid(True, ls=':', alpha=0.6)
            ax.legend(loc='best')
            fig.autofmt_xdate()
            out_pdf = os.path.join(root_dir, f"daily_rmse_{var}_{'_'.join(experiments)}_{start_date}_{end_date}.pdf")
            fig.savefig(out_pdf, bbox_inches="tight", dpi=150)
            print(f"Saved plot to {out_pdf}")
            plt.show()

        # optional save CSV per variable
        if save_csv_dir:
            os.makedirs(save_csv_dir, exist_ok=True)
            out_csv = os.path.join(save_csv_dir, f"daily_rmse_{var}_{'_'.join(experiments)}_{start_date}_{end_date}.csv")
            df_var.reset_index().rename(columns={"index": "date"}).to_csv(out_csv, index=False)
            print(f"Saved CSV to {out_csv}")

    overall_df = pd.DataFrame(overall_rows).pivot(index="experiment", columns="variable", values="overall_rmse")
    return all_daily, overall_df

# # Example usage:
# root = "/mimer/NOBACKUP/groups/mlhighres/users/mikhaili/neural-lam/output/230126"
# experiments = ["test_SI", "test_CorrDiff"]
# variables = ["pr", "tas"]
# start_date = "2010-01-01"
# end_date = "2014-12-31"
# all_daily, overall = multi_model_rmse_plot(root, experiments, variables, start_date, end_date,
#                                            save_csv_dir="output", plot=True)
# print(overall)

In [ ]:
# ...existing code...
import scipy.stats as stats

def _daily_target_stat_from_netcdf(target_nc: str, var_name: str, stat: str = "mean",
                                   start_date: Optional[str] = None, end_date: Optional[str] = None) -> pd.Series:
    """
    Return a pandas Series indexed by date (YYYY-MM-DD) with a daily spatial statistic
    (mean or median) computed from the target netCDF file.
    """
    ds = xr.open_dataset(target_nc)
    sel_slice = slice(start_date, end_date) if start_date or end_date else slice(None)
    ds = ds.sel(time=sel_slice)
    if var_name not in ds.data_vars:
        ds.close()
        raise RuntimeError(f"Variable {var_name} not in {target_nc}")
    # compute daily spatial stat (over non-time dims)
    arr = ds[var_name]
    # collapse spatial dims (any dims other than time)
    spatial_dims = [d for d in arr.dims if d != "time"]
    if stat == "mean":
        daily_vals = arr.mean(dim=spatial_dims, skipna=True)
    elif stat == "median":
        daily_vals = arr.median(dim=spatial_dims, skipna=True)
    else:
        ds.close()
        raise ValueError("stat must be 'mean' or 'median'")
    # to pandas Series (date string index)
    # robust conversion: handle DatetimeIndex, numpy datetime64, or object arrays
    times = np.asarray(daily_vals["time"].values)
    try:
        dates = np.datetime_as_string(times, unit="D")
    except Exception:
        # fallback to pandas string formatting
        dates = pd.to_datetime(times).strftime("%Y-%m-%d").tolist()
    s = pd.Series(daily_vals.values, index=dates)
    ds.close()
    return s

def plot_truth_vs_rmse(
    root_dir: str,
    all_daily: Dict[str, pd.DataFrame],
    experiments: List[str],
    variables: List[str],
    start_date: str,
    end_date: str,
    stat: str = "mean",
    figsize: tuple = (6, 4),
    save_dir: Optional[str] = None,
):
    """
    For each variable, load the daily target stat (mean/median) per experiment and
    scatter it against the daily RMSE (from all_daily). Produces one figure per variable
    which contains points for all experiments.
    """
    os.makedirs(save_dir, exist_ok=True) if save_dir else None

    for var in variables:
        if var not in all_daily:
            print(f"Skipping {var}: no RMSE data")
            continue
        df_rmse = all_daily[var].copy()
        if df_rmse.empty:
            print(f"No RMSE data for {var}")
            continue

        fig, ax = plt.subplots(figsize=figsize)
        colors = plt.cm.tab10.colors

        for i, exp in enumerate(experiments):
            # find target file for this experiment
            tgt_nc = _find_nc_file(root_dir, var, "target", exp)
            if tgt_nc is None:
                print(f"  Missing target for {var} / {exp}, skipping")
                continue
            try:
                s_truth = _daily_target_stat_from_netcdf(tgt_nc, var, stat=stat, start_date=start_date, end_date=end_date)
            except Exception as e:
                print(f"  Error reading target {tgt_nc}: {e}")
                continue

            # align with RMSE series for this experiment (may have different date coverage)
            if exp not in df_rmse.columns:
                print(f"  Experiment {exp} not present in RMSE table for {var}")
                continue
            s_rmse = df_rmse[exp].dropna()
            # join on dates
            joined = pd.DataFrame({"truth": s_truth, "rmse": s_rmse}).dropna()
            if joined.empty:
                print(f"  No overlapping dates for {var} / {exp}")
                continue

            c = colors[i % len(colors)]
            ax.scatter(joined["truth"], joined["rmse"], alpha=0.6, s=20, color=c, label=exp)
            # compute Pearson r
            try:
                r, p = stats.pearsonr(joined["truth"].values, joined["rmse"].values)
            except Exception:
                r, p = float("nan"), float("nan")
            ax.annotate(f"{exp}: r={r:.2f}", xy=(0.02, 0.95 - i*0.05), xycoords="axes fraction", color=c, fontsize=9)

        ax.set_xlabel(f"Daily target {stat} of {var}")
        ax.set_ylabel("Daily RMSE")
        ax.set_title(f"{var} truth vs RMSE ({stat})\n{start_date} to {end_date}")
        ax.grid(True, ls=":", alpha=0.5)
        ax.legend(loc="best")
        plt.tight_layout()

        if save_dir:
            out = os.path.join(save_dir, f"truth_vs_rmse_{var}_{stat}_{'_'.join(experiments)}_{start_date}_{end_date}.pdf")
            fig.savefig(out, bbox_inches="tight", dpi=150)
            print(f"Saved {out}")
        plt.show()

# Example call (after you obtain all_daily):
# plot_truth_vs_rmse(root, all_daily, experiments, ["pr","tas"], start_date, end_date, stat="mean", save_dir="output")
# ...existing code...

In [ ]:
def rmse_for_percentiles(
    all_daily: Dict[str, pd.DataFrame],
    root_dir: str,
    experiments: List[str],
    variables: List[str],
    percentiles: List[float],
    start_date: str,
    end_date: str,
    stat: str = "mean",
    mode: str = "above",   # "above" -> RMSE for days >= percentile; "bins" -> RMSE in percentile bins
    save_dir: Optional[str] = None,
    plot: bool = True,
) -> Dict[str, pd.DataFrame]:
    """
    Compute conditional RMSE for each variable / experiment for specified percentiles.

    - percentiles: list of percent values in (0..100), e.g. [50,90].
    - mode:
        * "above": RMSE for days where truth >= percentile threshold (per experiment).
        * "bins": RMSE computed in bins defined by consecutive percentile edges
                  (edges = [0] + percentiles + [100]).
    Returns dict var -> DataFrame (index experiments, columns percentile labels).
    """
    os.makedirs(save_dir, exist_ok=True) if save_dir else None
    out = {}

    for var in variables:
        if var not in all_daily:
            print(f"Skipping {var}: missing RMSE table")
            continue
        df_rmse = all_daily[var].copy()
        rows = []
        col_labels = []

        # prepare percentile edges
        pct_sorted = sorted(percentiles)
        if mode == "above":
            col_labels = [f">={int(p)}p" for p in pct_sorted]
        else:
            edges = [0.0] + pct_sorted + [100.0]
            col_labels = [f"{int(edges[i])}-{int(edges[i+1])}p" for i in range(len(edges)-1)]

        for exp in experiments:
            vals = {}
            # read truth daily stat for this experiment
            tgt_nc = _find_nc_file(root_dir, var, "target", exp)
            if tgt_nc is None:
                print(f"  Missing target for {var}/{exp}, skipping")
                for lab in col_labels:
                    vals[lab] = float("nan")
                rows.append(pd.Series(vals, name=exp))
                continue
            try:
                s_truth = _daily_target_stat_from_netcdf(tgt_nc, var, stat=stat, start_date=start_date, end_date=end_date)
            except Exception as e:
                print(f"  Error reading {tgt_nc}: {e}")
                for lab in col_labels:
                    vals[lab] = float("nan")
                rows.append(pd.Series(vals, name=exp))
                continue

            if exp not in df_rmse.columns:
                print(f"  RMSE for {exp} not available for {var}")
                for lab in col_labels:
                    vals[lab] = float("nan")
                rows.append(pd.Series(vals, name=exp))
                continue

            s_rmse = df_rmse[exp].dropna()
            joined = pd.DataFrame({"truth": s_truth, "rmse": s_rmse}).dropna()
            if joined.empty:
                for lab in col_labels:
                    vals[lab] = float("nan")
                rows.append(pd.Series(vals, name=exp))
                continue

            if mode == "above":
                for p in pct_sorted:
                    thr = np.nanpercentile(joined["truth"].values, p)
                    mask = joined["truth"] >= thr
                    vals[f">={int(p)}p"] = float(np.nanmean(joined.loc[mask, "rmse"].values)) if mask.any() else float("nan")
            else:  # bins
                edges = [0.0] + pct_sorted + [100.0]
                for i in range(len(edges)-1):
                    lo, hi = edges[i], edges[i+1]
                    lo_val = np.nanpercentile(joined["truth"].values, lo)
                    hi_val = np.nanpercentile(joined["truth"].values, hi)
                    if i == 0:
                        mask = (joined["truth"] >= lo_val) & (joined["truth"] <= hi_val)
                    else:
                        mask = (joined["truth"] > lo_val) & (joined["truth"] <= hi_val)
                    vals[f"{int(lo)}-{int(hi)}p"] = float(np.nanmean(joined.loc[mask, "rmse"].values)) if mask.any() else float("nan")

            rows.append(pd.Series(vals, name=exp))

        df_out = pd.DataFrame(rows)
        df_out.index.name = "experiment"
        out[var] = df_out

        if save_dir:
            out_csv = os.path.join(save_dir, f"rmse_by_percentile_{var}_{mode}_{'_'.join(str(int(p)) for p in percentiles)}.csv")
            df_out.reset_index().to_csv(out_csv, index=False)
            print(f"Saved percentile RMSE table to {out_csv}")

        if plot:
            fig, ax = plt.subplots(figsize=(max(6, len(col_labels)*1.2), 4))
            df_out.plot(kind="bar", ax=ax)
            ax.set_title(f"Conditional RMSE by percentile — {var} ({mode})\n{start_date} to {end_date}")
            ax.set_ylabel("RMSE")
            ax.grid(True, ls=":", alpha=0.4)
            plt.tight_layout()
            if save_dir:
                out_pdf = os.path.join(save_dir, f"rmse_by_percentile_{var}_{mode}_{'_'.join(str(int(p)) for p in percentiles)}.pdf")
                fig.savefig(out_pdf, bbox_inches="tight", dpi=150)
                print(f"Saved plot to {out_pdf}")
            plt.show()

    return out
# ...existing code...

# compute RMSE for top 90th and 95th percentiles (days with high truth values)
# pct_tables = rmse_for_percentiles(
#     all_daily, root, experiments, ["pr","tas"], percentiles=[90,95,99],
#     start_date=start_date, end_date=end_date, stat="mean",
#     mode="above", save_dir="output", plot=True
# )
# print(pct_tables["pr"])
# print(pct_tables["tas"])

In [ ]:
# ...existing code...
import torch
from tqdm import tqdm

def daily_and_overall_crps_from_netcdf(
    target_nc: str,
    var_name: Optional[str] = None,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    save_csv: Optional[str] = None,
    plot: bool = True,
    ensemble_member_ncs: Optional[List[str]] = None,
    root_dir: Optional[str] = None,
    exp: Optional[str] = None,
) -> Tuple[pd.Series, float]:
    """
    Compute daily CRPS using either:
      - ensemble member files -> metrics.crps_ens (preferred if members available), or
      - ensemble mean + std -> metrics.crps_gauss (fallback).

    Auto-discovers member files if ensemble_member_ncs is None and both root_dir and exp
    are provided. Member filename pattern expected like:
      <var>_ensemble_member_<i>_<EXP>_<YYYY-MM-DD>_<YYYY-MM-DD>.nc

    Uses GPU if available (torch.cuda.is_available()) by placing tensors on that device.
    """
    # device selection
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dtype = torch.get_default_dtype()  # usually torch.float32
    if device.type == "cuda":
        print(f"Using GPU device: {torch.cuda.get_device_name(0)}")
    else:
        print("Using CPU for CRPS computation")

    # try auto-discovery of ensemble member files when requested
    if ensemble_member_ncs is None and root_dir and exp:
        # if var_name not yet known, allow wildcard; members usually start with var name
        member_pattern = os.path.join(root_dir, exp, f"{var_name or '*'}*ensemble_member*.nc")
        found = sorted(glob.glob(member_pattern))
        if found:
            ensemble_member_ncs = found
            if var_name is None:
                # infer var_name from first member filename prefix (before first underscore)
                base = os.path.basename(found[0])
                var_name = base.split("_", 1)[0]

    # If we have ensemble members -> use crps_ens
    if ensemble_member_ncs is not None and len(ensemble_member_ncs) > 0:
        ds_tgt = xr.open_dataset(target_nc)

        # pick var if still not provided (from target)
        if var_name is None:
            data_vars = [v for v in ds_tgt.data_vars.keys()]
            if not data_vars:
                ds_tgt.close()
                raise RuntimeError("No data variables found in target file")
            var_name = data_vars[0]

        sel_slice = slice(start_date, end_date) if start_date or end_date else slice(None)
        ds_tgt = ds_tgt.sel(time=sel_slice)

        # open members (lazy) and align times
        ds_members = [xr.open_dataset(p).sel(time=sel_slice) for p in ensemble_member_ncs]
        # compute intersection of times across members and target
        times = np.asarray(ds_tgt["time"].values)
        for ds_m in ds_members:
            times = np.intersect1d(times, np.asarray(ds_m["time"].values))
        if times.size == 0:
            ds_tgt.close()
            for ds_m in ds_members:
                ds_m.close()
            raise RuntimeError("No overlapping times between target and ensemble member datasets")

        ds_tgt = ds_tgt.sel(time=times)
        ds_members = [ds.sel(time=times) for ds in ds_members]

        daily = {}
        for t in tqdm(ds_tgt.time.values):
            mem_vals = []
            for ds_m in ds_members:
                v = np.asarray(ds_m[var_name].sel(time=t).values).reshape((-1,))  # (N,)
                mem_vals.append(v)
            pred_np = np.stack(mem_vals, axis=0)  # (M, N)
            target_np = np.asarray(ds_tgt[var_name].sel(time=t).values).reshape((-1,))  # (N,)

            # convert to torch on chosen device: pred shape (1, M, N, 1), target shape (1, N, 1)
            pred_t = torch.as_tensor(pred_np, dtype=dtype, device=device).unsqueeze(-1).unsqueeze(0)
            targ_t = torch.as_tensor(target_np, dtype=dtype, device=device).unsqueeze(-1).unsqueeze(0)

            try:
                crps_val = metrics.crps_ens(pred_t, targ_t, None, mask=None, average_grid=True, sum_vars=True, ens_dim=1)
                crps_f = float(crps_val.detach().cpu().numpy())
            except Exception:
                crps_f = float("nan")

            date_str = np.datetime_as_string(t, unit="D")
            daily[date_str] = crps_f

        daily_series = pd.Series(daily).sort_index()

        # overall CRPS: use mean of daily CRPS values (ignores NaNs) to avoid stacking huge arrays
        overall = float(daily_series.mean()) if not daily_series.empty else float("nan")

        if save_csv:
            pd.DataFrame({"date": daily_series.index, "crps": daily_series.values}).to_csv(save_csv, index=False)

        if plot:
            dates = pd.to_datetime(daily_series.index)
            fig, ax = plt.subplots(figsize=(10, 4))
            ax.plot(dates, daily_series.values, marker="o", linestyle="-")
            ax.set_xlabel("Date")
            ax.set_ylabel("Daily CRPS (ens)")
            ax.set_title(f"Daily CRPS (ens) {times[0].astype('M8[D]').astype(str)} to {times[-1].astype('M8[D]').astype(str)}\nOverall CRPS={overall:.4f}")
            ax.grid(True, ls=":", alpha=0.6)
            fig.autofmt_xdate()
            plt.show()

        ds_tgt.close()
        for ds_m in ds_members:
            ds_m.close()
        return daily_series, overall


def daily_spread_from_netcdf(ensemble_std_nc: str, var_name: Optional[str] = None,
                             start_date: Optional[str] = None, end_date: Optional[str] = None,
                             save_csv: Optional[str] = None, plot: bool = True) -> Tuple[pd.Series, float]:
    """
    Compute daily RMS spread from ensemble_std netCDF (std per grid). Returns (daily_spread_series, overall_spread).
    daily_spread = sqrt(mean(std^2) over spatial grid)  -> RMS spread scalar per day.
    """
    ds_std = xr.open_dataset(ensemble_std_nc)
    if var_name is None:
        data_vars = [v for v in ds_std.data_vars.keys()]
        if not data_vars:
            raise RuntimeError("No data vars found in ensemble_std file")
        var_name = data_vars[0]

    sel_slice = slice(start_date, end_date) if start_date or end_date else slice(None)
    ds_std = ds_std.sel(time=sel_slice)

    times = ds_std["time"].values
    daily = {}
    for t in ds_std.time.values:
        s = np.asarray(ds_std[var_name].sel(time=t).values).ravel()  # per-grid std
        s2 = s**2
        if s2.size == 0:
            daily_val = float("nan")
        else:
            daily_val = float(np.sqrt(np.nanmean(s2)))
        date_str = np.datetime_as_string(t, unit="D")
        daily[date_str] = daily_val

    daily_series = pd.Series(daily).sort_index()
    # overall spread (RMS over all times & space)
    s_all = np.asarray(ds_std[var_name].values).ravel()
    overall = float(np.sqrt(np.nanmean(s_all**2))) if s_all.size else float("nan")

    if save_csv:
        pd.DataFrame({"date": daily_series.index, "spread": daily_series.values}).to_csv(save_csv, index=False)

    if plot:
        dates = pd.to_datetime(daily_series.index)
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(dates, daily_series.values, marker="o", linestyle="-")
        ax.set_xlabel("Date")
        ax.set_ylabel("RMS spread")
        ax.set_title(f"Daily RMS spread {dates[0].astype('M8[D]').astype(str)} to {dates[-1].astype('M8[D]').astype(str)}\nOverall spread={overall:.4f}")
        ax.grid(True, ls=":", alpha=0.6)
        fig.autofmt_xdate()
        plt.show()

    ds_std.close()
    return daily_series, overall

def spread_skill_ratio(
    all_daily_rmse: Dict[str, pd.DataFrame],
    root_dir: str,
    experiments: List[str],
    variables: List[str],
    start_date: str,
    end_date: str,
    save_dir: Optional[str] = None,
    plot: bool = True,
) -> Dict[str, pd.DataFrame]:
    """
    Compute spread / RMSE ratio per experiment and variable.
    Requires ensemble_std files to exist (tag 'ensemble_std' found by _find_nc_file).
    Returns dict var -> DataFrame (index experiments, columns: mean_ratio, median_ratio).
    """
    os.makedirs(save_dir, exist_ok=True) if save_dir else None
    out = {}

    for var in variables:
        print(f"Processing spread-skill ratio for variable '{var}'")
        rows = []
        for exp in experiments:
            print(f"  Experiment: {exp}")
            if var not in all_daily_rmse:
                vals = {"mean_ratio": float("nan"), "median_ratio": float("nan")}
                rows.append(pd.Series(vals, name=exp))
                continue

            # find files
            std_nc = _find_nc_file(root_dir, var, "ensemble_std", exp)
            if std_nc is None:
                print(f"  Missing ensemble_std for {var}/{exp}, skipping ratio")
                vals = {"mean_ratio": float("nan"), "median_ratio": float("nan")}
                rows.append(pd.Series(vals, name=exp))
                continue

            spread_series, _ = daily_spread_from_netcdf(std_nc, var_name=var, start_date=start_date, end_date=end_date, plot=False)
            rmse_series = all_daily_rmse[var].get(exp)
            if rmse_series is None:
                print(f"  Missing RMSE series for {var}/{exp}")
                vals = {"mean_ratio": float("nan"), "median_ratio": float("nan")}
                rows.append(pd.Series(vals, name=exp))
                continue

            joined = pd.DataFrame({"spread": spread_series, "rmse": rmse_series}).dropna()
            if joined.empty:
                vals = {"mean_ratio": float("nan"), "median_ratio": float("nan")}
            else:
                # avoid division by zero
                ratio = joined["spread"] / joined["rmse"].replace(0, np.nan)
                vals = {"mean_ratio": float(np.nanmean(ratio.values)), "median_ratio": float(np.nanmedian(ratio.values))}
            rows.append(pd.Series(vals, name=exp))

        df_out = pd.DataFrame(rows)
        df_out.index.name = "experiment"
        out[var] = df_out

        if save_dir:
            df_out.reset_index().to_csv(os.path.join(save_dir, f"spread_skill_ratio_{var}.csv"), index=False)

        if plot:
            fig, ax = plt.subplots(figsize=(6, 4))
            df_out[["mean_ratio", "median_ratio"]].plot(kind="bar", ax=ax)
            ax.set_title(f"Spread / RMSE ratio — {var}\n{start_date} to {end_date}")
            ax.set_ylabel("spread / RMSE")
            ax.grid(True, ls=":", alpha=0.4)
            plt.tight_layout()
            if save_dir:
                fig.savefig(os.path.join(save_dir, f"spread_skill_ratio_{var}.pdf"), bbox_inches="tight", dpi=150)
            plt.show()

    return out

In [ ]:
def compute_all_metrics_and_save_csv(
    root_dir: str,
    experiments: List[str],
    variables: List[str],
    start_date: str,
    end_date: str,
    out_dir: str = "output",
    plot: bool = False,
    use_gpu: bool = True,
) -> pd.DataFrame:
    """
    For each experiment and variable compute:
      - daily RMSE (daily_and_overall_rmse_from_netcdf)
      - daily spread (daily_spread_from_netcdf)
      - daily CRPS (daily_and_overall_crps_from_netcdf) — runs on GPU when available
      - daily target mean (_daily_target_stat_from_netcdf)

    Produces per-experiment/variable CSVs with columns:
      date, rmse, spread, ssr, crps, target_mean

    Also writes a summary CSV (one row per experiment/variable) with overall numbers
    and returns the summary DataFrame.

    Note: functions used above already handle missing files and date alignment.
    """
    os.makedirs(out_dir, exist_ok=True)
    summary_rows = []

    # Ensure GPU selection for CRPS function (it already checks torch.cuda.is_available())
    for exp in experiments:
        for var in variables:
            row = {
                "experiment": exp,
                "variable": var,
                "overall_rmse": float("nan"),
                "overall_spread": float("nan"),
                "overall_crps": float("nan"),
                "overall_target_mean": float("nan"),
                "csv_path": None,
            }

            mean_nc = _find_nc_file(root_dir, var, "ensemble_mean", exp)
            std_nc = _find_nc_file(root_dir, var, "ensemble_std", exp)
            tgt_nc = _find_nc_file(root_dir, var, "target", exp)

            if tgt_nc is None:
                print(f"Skipping {exp}/{var}: missing target file")
                summary_rows.append(row)
                continue

            # 1) daily RMSE
            try:
                if mean_nc is not None:
                    daily_rmse, overall_rmse = daily_and_overall_rmse_from_netcdf(
                        mean_nc,
                        tgt_nc,
                        var_name=var,
                        start_date=start_date,
                        end_date=end_date,
                        save_csv=None,
                        plot=False,
                    )
                else:
                    # no ensemble mean available -> fill with NaNs
                    daily_rmse = pd.Series(dtype=float)
                    overall_rmse = float("nan")
            except Exception as e:
                print(f"RMSE error for {exp}/{var}: {e}")
                daily_rmse = pd.Series(dtype=float)
                overall_rmse = float("nan")

            # 2) daily spread (RMS spread)
            try:
                if std_nc is not None:
                    daily_spread, overall_spread = daily_spread_from_netcdf(
                        std_nc, var_name=var, start_date=start_date, end_date=end_date, save_csv=None, plot=False
                    )
                else:
                    daily_spread = pd.Series(dtype=float)
                    overall_spread = float("nan")
            except Exception as e:
                print(f"Spread error for {exp}/{var}: {e}")
                daily_spread = pd.Series(dtype=float)
                overall_spread = float("nan")

            # 3) daily CRPS (auto-discovers members if present) — runs on GPU if available
            try:
                daily_crps, overall_crps = daily_and_overall_crps_from_netcdf(
                    target_nc=tgt_nc,
                    var_name=var,
                    start_date=start_date,
                    end_date=end_date,
                    save_csv=None,
                    plot=False,
                    root_dir=root_dir,
                    exp=exp,
                )
            except Exception as e:
                print(f"CRPS error for {exp}/{var}: {e}")
                daily_crps = pd.Series(dtype=float)
                overall_crps = float("nan")

            # 4) daily target mean
            try:
                target_mean_daily = _daily_target_stat_from_netcdf(
                    tgt_nc, var_name=var, stat="mean", start_date=start_date, end_date=end_date
                )
            except Exception as e:
                print(f"Target mean error for {exp}/{var}: {e}")
                target_mean_daily = pd.Series(dtype=float)

            # Align by date: inner join to keep days with data in any series; prefer index as date strings
            parts = {}
            if not daily_rmse.empty:
                parts["rmse"] = daily_rmse
            if not daily_spread.empty:
                parts["spread"] = daily_spread
            if not daily_crps.empty:
                parts["crps"] = daily_crps
            if not target_mean_daily.empty:
                parts["target_mean"] = target_mean_daily

            if parts:
                df = pd.concat(parts.values(), axis=1, keys=parts.keys())
                # ensure index is date strings
                df.index = df.index.astype(str)
                # compute SSR per day = spread / rmse (avoid divide by zero)
                if "spread" in df.columns and "rmse" in df.columns:
                    df["ssr"] = df["spread"] / df["rmse"].replace(0, np.nan)
                else:
                    df["ssr"] = np.nan
            else:
                df = pd.DataFrame(columns=["rmse", "spread", "crps", "target_mean", "ssr"])

            # order columns
            cols = ["rmse", "spread", "ssr", "crps", "target_mean"]
            for c in cols:
                if c not in df.columns:
                    df[c] = np.nan
            df = df[cols]

            out_csv = os.path.join(out_dir, f"metrics_{exp}_{var}_{start_date}_{end_date}.csv")
            df_reset = df.reset_index().rename(columns={"index": "date"})
            df_reset.to_csv(out_csv, index=False)
            row["csv_path"] = out_csv

            # fill overall summary values: prefer summary from functions when available, else compute from daily series
            row["overall_rmse"] = overall_rmse if not np.isnan(overall_rmse) else (float(df["rmse"].mean()) if not df["rmse"].dropna().empty else float("nan"))
            row["overall_spread"] = overall_spread if not np.isnan(overall_spread) else (float(df["spread"].mean()) if not df["spread"].dropna().empty else float("nan"))
            row["overall_crps"] = overall_crps if not np.isnan(overall_crps) else (float(df["crps"].mean()) if not df["crps"].dropna().empty else float("nan"))
            row["overall_target_mean"] = float(df["target_mean"].mean()) if not df["target_mean"].dropna().empty else float("nan")

            summary_rows.append(row)
            print(f"Saved metrics CSV for {exp}/{var} -> {out_csv}")

    summary_df = pd.DataFrame(summary_rows)
    summary_csv = os.path.join(out_dir, f"metrics_summary_{start_date}_{end_date}.csv")
    summary_df.to_csv(summary_csv, index=False)
    print(f"Saved summary CSV -> {summary_csv}")

    return summary_df

In [ ]:
import tempfile
# ...existing code...

def _create_mean_from_member_ncs_temp(member_ncs: List[str], preferred_var: str = "pr") -> Optional[str]:
    """Create a temporary netCDF with the ensemble mean computed from the provided member netCDFs.
    Returns path to temp file or None on failure.
    """
    if not member_ncs:
        return None
    try:
        # open first to infer var name and time coords
        ds0 = xr.open_dataset(member_ncs[0])
    except Exception:
        return None

    var_name = preferred_var if preferred_var in ds0.data_vars else (list(ds0.data_vars.keys())[0] if ds0.data_vars else None)
    if var_name is None:
        ds0.close()
        return None

    times = np.asarray(ds0["time"].values)
    sum_da = None
    count = 0

    for p in member_ncs:
        try:
            ds = xr.open_dataset(p)
            vn = var_name if var_name in ds.data_vars else (list(ds.data_vars.keys())[0] if ds.data_vars else None)
            if vn is None:
                ds.close()
                continue
            # select same times as first member (align)
            da = ds[vn].sel(time=times)
            # ensure numeric dtype to avoid unexpected types
            da = da.astype("float32")
            if sum_da is None:
                sum_da = da.copy(deep=True)
            else:
                sum_da = sum_da + da
            count += 1
            ds.close()
        except Exception:
            # skip problematic member
            continue

    ds0.close()

    if sum_da is None or count == 0:
        return None

    mean_da = sum_da / float(count)
    ds_out = mean_da.to_dataset(name=var_name)
    tf = tempfile.NamedTemporaryFile(prefix="mean_from_members_", suffix=".nc", delete=False)
    tf.close()
    outp = tf.name
    try:
        ds_out.to_netcdf(outp)
    finally:
        ds_out.close()
    return outp

def _clip_negative_precip_nc(nc_path: Optional[str], preferred_var: str = "pr") -> Optional[str]:
    """Return a path to a temporary netCDF where negative precipitation in preferred_var is set to 0.
    If no clipping needed or nc_path is None, returns the original nc_path.
    """
    if nc_path is None:
        return None
    try:
        ds = xr.open_dataset(nc_path)
    except Exception:
        return nc_path

    # infer variable name
    var_name = preferred_var if preferred_var in ds.data_vars else (list(ds.data_vars.keys())[0] if ds.data_vars else None)
    if var_name is None:
        ds.close()
        return nc_path

    try:
        vals = np.asarray(ds[var_name].values)
    except Exception:
        ds.close()
        return nc_path

    # if no negative values -> keep original file
    if not np.any(np.isfinite(vals) & (vals < 0.0)):
        ds.close()
        return nc_path

    # create clipped dataset and write to temp file
    try:
        ds_clip = ds.copy()
        ds_clip[var_name] = ds[var_name].where(ds[var_name] >= 0, other=0.0)
        tf = tempfile.NamedTemporaryFile(prefix="clip_pr_", suffix=".nc", delete=False)
        tf.close()
        outp = tf.name
        ds_clip.to_netcdf(outp)
        ds.close()
        ds_clip.close()
        return outp
    except Exception:
        try:
            ds.close()
        except Exception:
            pass
        return nc_path

def compute_all_metrics_and_save_csv(
    root_dir: str,
    experiments: List[str],
    variables: List[str],
    start_date: str,
    end_date: str,
    out_dir: str = "output",
    plot: bool = False,
    use_gpu: bool = True,
    clip_negative_pr: bool = False,   # NEW: when True, set negative precipitation in model predictions -> 0
) -> pd.DataFrame:
    # ...existing code...
    os.makedirs(out_dir, exist_ok=True)
    summary_rows = []

    for exp in experiments:
        for var in variables:
            row = {
                "experiment": exp,
                "variable": var,
                "overall_rmse": float("nan"),
                "overall_spread": float("nan"),
                "overall_crps": float("nan"),
                "overall_target_mean": float("nan"),
                "csv_path": None,
            }
            mean_nc = _find_nc_file(root_dir, var, "ensemble_mean", exp)
            std_nc = _find_nc_file(root_dir, var, "ensemble_std", exp)
            tgt_nc = _find_nc_file(root_dir, var, "target", exp)

            if tgt_nc is None:
                print(f"Skipping {exp}/{var}: missing target file")
                summary_rows.append(row)
                continue

            # If requested, clip negative precipitation in model outputs (members + mean).
            # Important: if members exist, recompute ensemble mean from clipped members so mean reflects clipping.
            clipped_mean_nc = None
            clipped_member_ncs = None
            temp_mean_from_members = None
            if clip_negative_pr and var == "pr":
                # Discover member files first
                member_files = _collect_member_files(root_dir, exp, var)
                if member_files:
                    # Clip each member (writes temp files only for members that needed clipping)
                    clipped_member_ncs = [ _clip_negative_precip_nc(mf, preferred_var=var) for mf in member_files ]
                    # Recompute ensemble mean from clipped members (prefer clipped member files where they exist)
                    # Use clipped_member_ncs (which may contain original paths if no clipping was needed)
                    temp_mean_from_members = _create_mean_from_member_ncs_temp(clipped_member_ncs, preferred_var=var)
                    if temp_mean_from_members:
                        mean_nc_to_use = temp_mean_from_members
                    else:
                        # fallback: clip existing ensemble mean file if present
                        mean_nc_to_use = _clip_negative_precip_nc(mean_nc, preferred_var=var) if mean_nc is not None else None
                else:
                    # no members found: clip existing mean if present (no recompute possible)
                    if mean_nc is not None:
                        clipped_mean_nc = _clip_negative_precip_nc(mean_nc, preferred_var=var)
                        mean_nc_to_use = clipped_mean_nc if clipped_mean_nc else mean_nc
                    else:
                        mean_nc_to_use = None
            else:
                mean_nc_to_use = mean_nc
                clipped_member_ncs = None

            # 1) daily RMSE
            try:
                if mean_nc_to_use is not None:
                    daily_rmse, overall_rmse = daily_and_overall_rmse_from_netcdf(
                        mean_nc_to_use,
                        tgt_nc,
                        var_name=var,
                        start_date=start_date,
                        end_date=end_date,
                        save_csv=None,
                        plot=False,
                    )
                else:
                    daily_rmse = pd.Series(dtype=float)
                    overall_rmse = float("nan")
            except Exception as e:
                print(f"RMSE error for {exp}/{var}: {e}")
                daily_rmse = pd.Series(dtype=float)
                overall_rmse = float("nan")

            # 2) daily spread (RMS spread)
            try:
                if std_nc is not None:
                    daily_spread, overall_spread = daily_spread_from_netcdf(
                        std_nc, var_name=var, start_date=start_date, end_date=end_date, save_csv=None, plot=False
                    )
                else:
                    daily_spread = pd.Series(dtype=float)
                    overall_spread = float("nan")
            except Exception as e:
                print(f"Spread error for {exp}/{var}: {e}")
                daily_spread = pd.Series(dtype=float)
                overall_spread = float("nan")

            # 3) daily CRPS (auto-discovers members if present) — runs on GPU if available
            try:
                # if we clipped members, pass them explicitly to CRPS function; otherwise let it auto-discover
                daily_crps, overall_crps = daily_and_overall_crps_from_netcdf(
                    target_nc=tgt_nc,
                    var_name=var,
                    start_date=start_date,
                    end_date=end_date,
                    save_csv=None,
                    plot=False,
                    root_dir=root_dir,
                    exp=exp,
                    ensemble_member_ncs=clipped_member_ncs if clipped_member_ncs is not None else None,
                )
            except Exception as e:
                print(f"CRPS error for {exp}/{var}: {e}")
                daily_crps = pd.Series(dtype=float)
                overall_crps = float("nan")

            # 4) daily target mean
            try:
                target_mean_daily = _daily_target_stat_from_netcdf(
                    tgt_nc, var_name=var, stat="mean", start_date=start_date, end_date=end_date
                )
            except Exception as e:
                print(f"Target mean error for {exp}/{var}: {e}")
                target_mean_daily = pd.Series(dtype=float)

            # Align by date: inner join to keep days with data in any series; prefer index as date strings
            parts = {}
            if not daily_rmse.empty:
                parts["rmse"] = daily_rmse
            if not daily_spread.empty:
                parts["spread"] = daily_spread
            if not daily_crps.empty:
                parts["crps"] = daily_crps
            if not target_mean_daily.empty:
                parts["target_mean"] = target_mean_daily

            if parts:
                df = pd.concat(parts.values(), axis=1, keys=parts.keys())
                # ensure index is date strings
                df.index = df.index.astype(str)
                # compute SSR per day = spread / rmse (avoid divide by zero)
                if "spread" in df.columns and "rmse" in df.columns:
                    df["ssr"] = df["spread"] / df["rmse"].replace(0, np.nan)
                else:
                    df["ssr"] = np.nan
            else:
                df = pd.DataFrame(columns=["rmse", "spread", "ssr", "crps", "target_mean"])

            # order columns
            cols = ["rmse", "spread", "ssr", "crps", "target_mean"]
            for c in cols:
                if c not in df.columns:
                    df[c] = np.nan
            df = df[cols]

            out_csv = os.path.join(out_dir, f"metrics_{exp}_{var}_{start_date}_{end_date}.csv")
            df_reset = df.reset_index().rename(columns={"index": "date"})
            df_reset.to_csv(out_csv, index=False)
            row["csv_path"] = out_csv

            # fill overall summary values
            row["overall_rmse"] = overall_rmse if not np.isnan(overall_rmse) else (float(df["rmse"].mean()) if not df["rmse"].dropna().empty else float("nan"))
            row["overall_spread"] = overall_spread if not np.isnan(overall_spread) else (float(df["spread"].mean()) if not df["spread"].dropna().empty else float("nan"))
            row["overall_crps"] = overall_crps if not np.isnan(overall_crps) else (float(df["crps"].mean()) if not df["crps"].dropna().empty else float("nan"))
            row["overall_target_mean"] = float(df["target_mean"].mean()) if not df["target_mean"].dropna().empty else float("nan")

            summary_rows.append(row)
            print(f"Saved metrics CSV for {exp}/{var} -> {out_csv}")

    summary_df = pd.DataFrame(summary_rows)
    summary_csv = os.path.join(out_dir, f"metrics_summary_{start_date}_{end_date}.csv")
    summary_df.to_csv(summary_csv, index=False)
    print(f"Saved summary CSV -> {summary_csv}")

    return summary_df

In [ ]:
# import pandas as pd
# import xarray as xr
# import numpy as np
# import os
# from typing import List, Optional, Tuple

# # ...existing code...
# def compute_negative_precip_stats(
#     root_dir: str,
#     experiments: List[str],
#     var: str = "pr",
#     start_date: Optional[str] = None,
#     end_date: Optional[str] = None,
#     max_members: Optional[int] = None,
#     out_csv: Optional[str] = None,
#     verbose: bool = True,
# ) -> Tuple[pd.DataFrame, pd.DataFrame]:
#     """
#     Walk ensemble member files for given experiments and variable and compute:
#       - number of pixels < 0 (count) across all times
#       - total negative precipitation (sum of negative values) across all times

#     Returns two DataFrames:
#       - per-member rows: experiment, member_file, n_time_steps, n_pixels_total, n_negative_pixels, total_negative_amount
#       - per-experiment summary: experiment, n_members, n_time_steps_total, n_pixels_total, n_negative_pixels, total_negative_amount

#     Notes:
#       - falls back to ensemble_mean file if no member files found (treated as single-member).
#       - uses same time selection logic as other functions (slice start_date..end_date).
#     """
#     rows = []
#     for exp in experiments:
#         member_files = _collect_member_files(root_dir, exp, var)
#         # fallback to ensemble_mean if no members
#         if not member_files:
#             mean_nc = _find_nc_file(root_dir, var, "ensemble_mean", exp)
#             if mean_nc:
#                 member_files = [mean_nc]
#         if max_members:
#             member_files = member_files[:max_members]
#         if verbose:
#             print(f"Experiment {exp}: {len(member_files)} member/mean files to scan")

#         exp_counts = 0
#         exp_pixels = 0
#         exp_neg_pixels = 0
#         exp_neg_sum = 0.0
#         exp_time_steps = 0

#         for mf in member_files:
#             try:
#                 ds = xr.open_dataset(mf)
#             except Exception as e:
#                 if verbose:
#                     print(f"  Cannot open {mf}: {e}")
#                 continue

#             # infer variable name
#             var_name = var if var in ds.data_vars else (list(ds.data_vars.keys())[0] if ds.data_vars else None)
#             if var_name is None:
#                 ds.close()
#                 if verbose:
#                     print(f"  No data var in {mf}, skipping")
#                 continue

#             sel_slice = slice(start_date, end_date) if start_date or end_date else slice(None)
#             try:
#                 da = ds[var_name].sel(time=sel_slice)
#             except Exception:
#                 ds.close()
#                 if verbose:
#                     print(f"  Cannot select time range in {mf}, skipping")
#                 continue

#             n_time = 0
#             n_pixels_total = 0
#             n_neg_pixels = 0
#             neg_sum = 0.0

#             for t in da.time.values:
#                 arr = np.asarray(da.sel(time=t).values).ravel()
#                 # ignore non-finite
#                 mask_f = np.isfinite(arr)
#                 if not mask_f.any():
#                     continue
#                 arr = arr[mask_f]
#                 n_time += 1
#                 n_pixels_total += arr.size
#                 neg_mask = arr < 0.0
#                 if neg_mask.any():
#                     nneg = int(neg_mask.sum())
#                     sneg = float(arr[neg_mask].sum())  # negative number (sum of negatives)
#                 else:
#                     nneg = 0
#                     sneg = 0.0
#                 n_neg_pixels += nneg
#                 neg_sum += sneg

#             ds.close()

#             rows.append({
#                 "experiment": exp,
#                 "member_file": os.path.basename(mf),
#                 "n_time_steps": int(n_time),
#                 "n_pixels_total": int(n_pixels_total),
#                 "n_negative_pixels": int(n_neg_pixels),
#                 "total_negative_amount": float(neg_sum),  # will be <= 0
#             })

#             # accumulate experiment totals
#             exp_counts += 1
#             exp_pixels += n_pixels_total
#             exp_neg_pixels += n_neg_pixels
#             exp_neg_sum += neg_sum
#             exp_time_steps += n_time

#         # per-experiment summary
#         summary = {
#             "experiment": exp,
#             "n_members": int(exp_counts),
#             "n_time_steps_total": int(exp_time_steps),
#             "n_pixels_total": int(exp_pixels),
#             "n_negative_pixels": int(exp_neg_pixels),
#             "total_negative_amount": float(exp_neg_sum),
#         }
#         # append a sentinel row to rows? better keep separate summary table
#         if verbose:
#             print(f"  Summary {exp}: members={summary['n_members']}, time_steps={summary['n_time_steps_total']}, neg_pixels={summary['n_negative_pixels']}, neg_sum={summary['total_negative_amount']:.3f}")

#     df_members = pd.DataFrame(rows)
#     df_summary = pd.DataFrame([{
#         "experiment": r["experiment"],
#         "n_members": (df_members[df_members["experiment"]==r["experiment"]].shape[0]),
#         "n_time_steps_total": int(df_members[df_members["experiment"]==r["experiment"]]["n_time_steps"].sum()),
#         "n_pixels_total": int(df_members[df_members["experiment"]==r["experiment"]]["n_pixels_total"].sum()),
#         "n_negative_pixels": int(df_members[df_members["experiment"]==r["experiment"]]["n_negative_pixels"].sum()),
#         "total_negative_amount": float(df_members[df_members["experiment"]==r["experiment"]]["total_negative_amount"].sum()),
#     } for r in df_members.drop_duplicates(subset=["experiment"]).to_dict("records")])

#     if out_csv:
#         df_members.to_csv(out_csv.replace(".csv", "_members.csv") if out_csv.endswith(".csv") else out_csv + "_members.csv", index=False)
#         df_summary.to_csv(out_csv.replace(".csv", "_summary.csv") if out_csv.endswith(".csv") else out_csv + "_summary.csv", index=False)
#         if verbose:
#             print(f"Saved member-level and summary CSVs to {out_csv}")

#     return df_members, df_summary

# # Example usage (run in notebook afterwards):
# df_members, df_summary = compute_negative_precip_stats(root, experiments, var="pr", start_date="2010-01-01", end_date="2014-12-31", max_members=50, out_csv="output/negative_pr_stats.csv")
# print(df_summary)

In [ ]:
# ...existing code...
import os
import tempfile
from typing import List, Optional

import xarray as xr
import numpy as np
import pandas as pd
# ...existing code...

def _create_mean_from_member_ncs_temp(member_ncs: List[str], preferred_var: str = "pr", tmp_dir: Optional[str] = None) -> Optional[str]:
    """Create a temporary netCDF with the ensemble mean computed from the provided member netCDFs.
    Returns path to temp file or None on failure. tmp_dir if provided will be used for the temp file.
    """
    if not member_ncs:
        return None
    try:
        ds0 = xr.open_dataset(member_ncs[0])
    except Exception:
        return None

    var_name = preferred_var if preferred_var in ds0.data_vars else (list(ds0.data_vars.keys())[0] if ds0.data_vars else None)
    if var_name is None:
        ds0.close()
        return None

    times = np.asarray(ds0["time"].values)
    sum_da = None
    count = 0

    for p in member_ncs:
        try:
            ds = xr.open_dataset(p)
            vn = var_name if var_name in ds.data_vars else (list(ds.data_vars.keys())[0] if ds.data_vars else None)
            if vn is None:
                ds.close()
                continue
            # select same times as first member (align)
            da = ds[vn].sel(time=times)
            # ensure numeric dtype to avoid unexpected types
            da = da.astype("float32")
            if sum_da is None:
                sum_da = da.copy(deep=True)
            else:
                sum_da = sum_da + da
            count += 1
            ds.close()
        except Exception:
            # skip problematic member
            continue

    ds0.close()

    if sum_da is None or count == 0:
        return None

    mean_da = sum_da / float(count)
    ds_out = mean_da.to_dataset(name=var_name)

    # choose temporary directory (prefer provided tmp_dir / out_dir over system /tmp)
    tmp_dir = tmp_dir or tempfile.gettempdir()
    tf = tempfile.NamedTemporaryFile(prefix="mean_from_members_", suffix=".nc", delete=False, dir=tmp_dir)
    tf.close()
    outp = tf.name
    try:
        ds_out.to_netcdf(outp)
        try:
            os.chmod(outp, 0o666)
        except Exception:
            pass
    finally:
        ds_out.close()
    return outp

def _clip_negative_precip_nc(nc_path: Optional[str], preferred_var: str = "pr", tmp_dir: Optional[str] = None) -> Optional[str]:
    """Return a path to a temporary netCDF where negative precipitation in preferred_var is set to 0.
    If no clipping needed or nc_path is None, returns the original nc_path. tmp_dir chooses where temp is created.
    """
    if nc_path is None:
        return None
    try:
        ds = xr.open_dataset(nc_path)
    except Exception:
        return nc_path

    var_name = preferred_var if preferred_var in ds.data_vars else (list(ds.data_vars.keys())[0] if ds.data_vars else None)
    if var_name is None:
        ds.close()
        return nc_path

    try:
        vals = np.asarray(ds[var_name].values)
    except Exception:
        ds.close()
        return nc_path

    # if no negative values -> keep original file
    if not np.any(np.isfinite(vals) & (vals < 0.0)):
        ds.close()
        return nc_path

    # create clipped dataset and write to temp file
    try:
        ds_clip = ds.copy()
        ds_clip[var_name] = ds[var_name].where(ds[var_name] >= 0, other=0.0)
        tmp_dir = tmp_dir or tempfile.gettempdir()
        tf = tempfile.NamedTemporaryFile(prefix="clip_pr_", suffix=".nc", delete=False, dir=tmp_dir)
        tf.close()
        outp = tf.name
        ds_clip.to_netcdf(outp)
        try:
            os.chmod(outp, 0o666)
        except Exception:
            pass
        ds.close()
        ds_clip.close()
        return outp
    except Exception:
        try:
            ds.close()
        except Exception:
            pass
        return nc_path

def compute_all_metrics_and_save_csv(
    root_dir: str,
    experiments: List[str],
    variables: List[str],
    start_date: str,
    end_date: str,
    out_dir: str = "output",
    plot: bool = False,
    use_gpu: bool = True,
    clip_negative_pr: bool = False,   # when True, set negative precipitation in model predictions -> 0
) -> pd.DataFrame:
    """
    For each experiment and variable compute:
      - daily RMSE
      - daily spread
      - daily CRPS (runs on GPU when available)
      - daily target mean

    Produces per-experiment/variable CSVs and a summary CSV. When clip_negative_pr=True and var=='pr'
    this will clip negatives in members and recompute the ensemble mean from those clipped members (if members are found).
    Temporary files are created inside out_dir to avoid /tmp permission issues.
    """
    os.makedirs(out_dir, exist_ok=True)
    summary_rows = []

    for exp in experiments:
        for var in variables:
            row = {
                "experiment": exp,
                "variable": var,
                "overall_rmse": float("nan"),
                "overall_spread": float("nan"),
                "overall_crps": float("nan"),
                "overall_target_mean": float("nan"),
                "csv_path": None,
            }

            mean_nc = _find_nc_file(root_dir, var, "ensemble_mean", exp)
            std_nc = _find_nc_file(root_dir, var, "ensemble_std", exp)
            tgt_nc = _find_nc_file(root_dir, var, "target", exp)

            if tgt_nc is None:
                print(f"Skipping {exp}/{var}: missing target file")
                summary_rows.append(row)
                continue

            # If requested, clip negative precipitation in model outputs (members + mean).
            # Important: if members exist, recompute ensemble mean from clipped members so mean reflects clipping.
            clipped_member_ncs = None
            temp_mean_from_members = None
            mean_nc_to_use = mean_nc

            if clip_negative_pr and var == "pr":
                # Discover member files first
                member_files = _collect_member_files(root_dir, exp, var)
                if member_files:
                    # Clip each member (writes temp files only for members that needed clipping)
                    clipped_member_ncs = [ _clip_negative_precip_nc(mf, preferred_var=var, tmp_dir=out_dir) for mf in member_files ]
                    # Recompute ensemble mean from clipped members (prefer clipped member files where they exist)
                    # Use clipped_member_ncs (which may contain original paths if no clipping was needed)
                    temp_mean_from_members = _create_mean_from_member_ncs_temp(clipped_member_ncs, preferred_var=var, tmp_dir=out_dir)
                    if temp_mean_from_members:
                        mean_nc_to_use = temp_mean_from_members
                    else:
                        # fallback: clip existing ensemble mean file if present
                        mean_nc_to_use = _clip_negative_precip_nc(mean_nc, preferred_var=var, tmp_dir=out_dir) if mean_nc is not None else None
                else:
                    # no members found: clip existing mean if present (no recompute possible)
                    if mean_nc is not None:
                        clipped_mean_nc = _clip_negative_precip_nc(mean_nc, preferred_var=var, tmp_dir=out_dir)
                        mean_nc_to_use = clipped_mean_nc if clipped_mean_nc else mean_nc
                    else:
                        mean_nc_to_use = None
            else:
                mean_nc_to_use = mean_nc
                clipped_member_ncs = None

            # 1) daily RMSE
            try:
                if mean_nc_to_use is not None:
                    daily_rmse, overall_rmse = daily_and_overall_rmse_from_netcdf(
                        mean_nc_to_use,
                        tgt_nc,
                        var_name=var,
                        start_date=start_date,
                        end_date=end_date,
                        save_csv=None,
                        plot=False,
                    )
                else:
                    # no ensemble mean available -> fill with NaNs
                    daily_rmse = pd.Series(dtype=float)
                    overall_rmse = float("nan")
            except Exception as e:
                print(f"RMSE error for {exp}/{var}: {e}")
                daily_rmse = pd.Series(dtype=float)
                overall_rmse = float("nan")

            # 2) daily spread (RMS spread)
            try:
                if std_nc is not None:
                    daily_spread, overall_spread = daily_spread_from_netcdf(
                        std_nc, var_name=var, start_date=start_date, end_date=end_date, save_csv=None, plot=False
                    )
                else:
                    daily_spread = pd.Series(dtype=float)
                    overall_spread = float("nan")
            except Exception as e:
                print(f"Spread error for {exp}/{var}: {e}")
                daily_spread = pd.Series(dtype=float)
                overall_spread = float("nan")

            # 3) daily CRPS (auto-discovers members if present) — runs on GPU if available
            try:
                # if we clipped members, pass them explicitly to CRPS function; otherwise let it auto-discover
                daily_crps, overall_crps = daily_and_overall_crps_from_netcdf(
                    target_nc=tgt_nc,
                    var_name=var,
                    start_date=start_date,
                    end_date=end_date,
                    save_csv=None,
                    plot=False,
                    root_dir=root_dir,
                    exp=exp,
                    ensemble_member_ncs=clipped_member_ncs if clipped_member_ncs is not None else None,
                )
            except Exception as e:
                print(f"CRPS error for {exp}/{var}: {e}")
                daily_crps = pd.Series(dtype=float)
                overall_crps = float("nan")

            # 4) daily target mean
            try:
                target_mean_daily = _daily_target_stat_from_netcdf(
                    tgt_nc, var_name=var, stat="mean", start_date=start_date, end_date=end_date
                )
            except Exception as e:
                print(f"Target mean error for {exp}/{var}: {e}")
                target_mean_daily = pd.Series(dtype=float)

            # Align by date: inner join to keep days with data in any series; prefer index as date strings
            parts = {}
            if not daily_rmse.empty:
                parts["rmse"] = daily_rmse
            if not daily_spread.empty:
                parts["spread"] = daily_spread
            if not daily_crps.empty:
                parts["crps"] = daily_crps
            if not target_mean_daily.empty:
                parts["target_mean"] = target_mean_daily

            if parts:
                df = pd.concat(parts.values(), axis=1, keys=parts.keys())
                # ensure index is date strings
                df.index = df.index.astype(str)
                # compute SSR per day = spread / rmse (avoid divide by zero)
                if "spread" in df.columns and "rmse" in df.columns:
                    df["ssr"] = df["spread"] / df["rmse"].replace(0, np.nan)
                else:
                    df["ssr"] = np.nan
            else:
                df = pd.DataFrame(columns=["rmse", "spread", "ssr", "crps", "target_mean"])

            # order columns
            cols = ["rmse", "spread", "ssr", "crps", "target_mean"]
            for c in cols:
                if c not in df.columns:
                    df[c] = np.nan
            df = df[cols]

            out_csv = os.path.join(out_dir, f"metrics_{exp}_{var}_{start_date}_{end_date}.csv")
            df_reset = df.reset_index().rename(columns={"index": "date"})
            df_reset.to_csv(out_csv, index=False)
            row["csv_path"] = out_csv

            # fill overall summary values
            row["overall_rmse"] = overall_rmse if not np.isnan(overall_rmse) else (float(df["rmse"].mean()) if not df["rmse"].dropna().empty else float("nan"))
            row["overall_spread"] = overall_spread if not np.isnan(overall_spread) else (float(df["spread"].mean()) if not df["spread"].dropna().empty else float("nan"))
            row["overall_crps"] = overall_crps if not np.isnan(overall_crps) else (float(df["crps"].mean()) if not df["crps"].dropna().empty else float("nan"))
            row["overall_target_mean"] = float(df["target_mean"].mean()) if not df["target_mean"].dropna().empty else float("nan")

            summary_rows.append(row)
            print(f"Saved metrics CSV for {exp}/{var} -> {out_csv}")

    summary_df = pd.DataFrame(summary_rows)
    summary_csv = os.path.join(out_dir, f"metrics_summary_{start_date}_{end_date}.csv")
    summary_df.to_csv(summary_csv, index=False)
    print(f"Saved summary CSV -> {summary_csv}")

    return summary_df

In [ ]:
# run (will auto-discover ensemble members, use GPU if available)
summary_df = compute_all_metrics_and_save_csv(
    root_dir=root,
    experiments=experiments,
    variables=variables,
    start_date=start_date,
    end_date=end_date,
    out_dir=output_dir,
    plot=False,
    use_gpu=True,
    clip_negative_pr=False,   # clip negative precipitation values to 0
)

print(summary_df)

In [ ]:
# load_metrics_means (paste into your notebook)
import os, glob, re
import pandas as pd
from pathlib import Path

out_dir = output_dir
pat = os.path.join(out_dir, "metrics_*.csv")
files = sorted(glob.glob(pat))

rows = []
rx = re.compile(r"metrics_(?P<exp>.*)_(?P<var>[^_]+)_(?P<start>\d{4}-\d{2}-\d{2})_(?P<end>\d{4}-\d{2}-\d{2})\.csv$")

for f in files:
    m = rx.search(os.path.basename(f))
    if not m:
        # fallback: use filename only
        exp = var = Path(f).stem
    else:
        exp = m.group("exp")
        var = m.group("var")
    df = pd.read_csv(f)
    # safe column names (if missing, fill NaN)
    for col in ["rmse","spread","ssr","crps","target_mean"]:
        if col not in df.columns:
            df[col] = pd.NA
    means = df[["rmse","spread","ssr","crps","target_mean"]].mean(skipna=True)
    rows.append({
        "experiment": exp,
        "variable": var,
        "mean_rmse": float(means["rmse"]) if pd.notna(means["rmse"]) else float("nan"),
        "mean_spread": float(means["spread"]) if pd.notna(means["spread"]) else float("nan"),
        "mean_ssr": float(means["ssr"]) if pd.notna(means["ssr"]) else float("nan"),
        "mean_crps": float(means["crps"]) if pd.notna(means["crps"]) else float("nan"),
        "mean_target": float(means["target_mean"]) if pd.notna(means["target_mean"]) else float("nan"),
        "csv": f
    })

summary = pd.DataFrame(rows).sort_values(["experiment","variable"]).reset_index(drop=True)
print(summary.to_string(index=False))

In [ ]:
# Plot timeseries of each metric for all experiments (add as a new cell)
import os, glob, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

out_dir = output_dir
pat = os.path.join(out_dir, "metrics_*.csv")
files = sorted(glob.glob(pat))
if not files:
    raise RuntimeError(f"No metrics CSVs found at {pat}")

# load all files into one long DataFrame (add experiment/variable from filename)
rx = re.compile(r"metrics_(?P<exp>.*)_(?P<var>[^_]+)_(?P<start>\d{4}-\d{2}-\d{2})_(?P<end>\d{4}-\d{2}-\d{2})\.csv$")
rows = []
for f in files:
    m = rx.search(os.path.basename(f))
    if m:
        exp = m.group("exp")
        var = m.group("var")
    else:
        exp = var = Path(f).stem
    df = pd.read_csv(f)
    # ensure required cols exist
    for col in ["date","rmse","spread","ssr","crps","target_mean"]:
        if col not in df.columns:
            df[col] = np.nan
    df["experiment"] = exp
    df["variable"] = var
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    rows.append(df[["date","experiment","variable","rmse","spread","ssr","crps","target_mean"]])

long = pd.concat(rows, ignore_index=True).dropna(subset=["date"])
metrics = ["rmse","spread","ssr","crps","target_mean"]

os.makedirs(os.path.join(out_dir,"plots"), exist_ok=True)

# one figure per variable, subplots per metric
for var in sorted(long["variable"].dropna().unique()):
    sub = long[long["variable"] == var].copy()
    if sub.empty:
        continue
    for metric in metrics:
        piv = sub.pivot(index="date", columns="experiment", values=metric)
        fig, ax = plt.subplots(figsize=(10, 4))
        if piv.dropna(how="all").empty:
            ax.text(0.5, 0.5, f"No data for {metric}", ha="center", va="center")
            ax.set_ylabel(metric)
        else:
            for col in piv.columns:
                if "CorrDiff" in col:
                    label="CorrDiff"
                elif "UNet" in col:
                    label="UNET"
                elif "SI" in col:
                    label="CDSI"
                else:
                    label=col
                ax.plot(piv.index, piv[col], marker="o", linestyle="-", label=label)
            ax.set_ylabel(metric)
            ax.grid(True, ls=":", alpha=0.5)
            ax.legend(loc="best", fontsize="large")
        ax.set_xlabel("Date")
        var_name = "Precipitation" if var == "pr" else "Temperature" if var == "tas" else var
        ax.set_title(f"{var_name} — {metric} over time")
        plt.tight_layout()
        out_pdf = os.path.join(out_dir, "plots", f"metrics_timeseries_{var}_{metric}.pdf")
        fig.savefig(out_pdf, bbox_inches="tight", dpi=150)
        print("Saved", out_pdf)
        plt.show()
        plt.close(fig)

In [ ]:
# Analyze extremes: load metrics CSVs, compute conditional RMSE for high percentiles,
# compute correlation rmse vs truth, and produce simple plots.
import os, glob, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

out_dir = output_dir
pat = os.path.join(out_dir, "metrics_*.csv")
files = sorted(glob.glob(pat))
if not files:
    raise RuntimeError(f"No metrics CSVs found at {pat}")

# Build long table with per-day metrics
rows = []
rx = re.compile(r"metrics_(?P<exp>.*)_(?P<var>[^_]+)_(?P<start>\d{4}-\d{2}-\d{2})_(?P<end>\d{4}-\d{2}-\d{2})\.csv$")
for f in files:
    m = rx.search(os.path.basename(f))
    if m:
        exp = m.group("exp")
        var = m.group("var")
    else:
        exp = var = Path(f).stem
    df = pd.read_csv(f)
    for col in ["rmse","spread","ssr","crps","target_mean","date"]:
        if col not in df.columns:
            df[col] = np.nan
    df["experiment"] = exp
    df["variable"] = var
    # ensure numeric
    df["rmse"] = pd.to_numeric(df["rmse"], errors="coerce")
    df["target_mean"] = pd.to_numeric(df["target_mean"], errors="coerce")
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    rows.append(df[["date","experiment","variable","rmse","spread","ssr","crps","target_mean"]])

long = pd.concat(rows, ignore_index=True).dropna(subset=["date"]).sort_values(["variable","experiment","date"])
print("Loaded rows:", len(long))

# Quick overview: mean metrics per experiment/variable
overview = long.groupby(["experiment","variable"])[["rmse","spread","ssr","crps","target_mean"]].mean().reset_index()
print("\nMean metrics (per experiment / variable):")
print(overview.to_string(index=False))

# Examine extremes: conditional RMSE for thresholds (per-experiment, per-variable)
thresholds = [90, 95, 99]
metrics_to_analyze = ["rmse", "ssr", "crps"]

# Build conditional tables and correlation tables for each metric
ext_tables = {}
corr_tables = {}
for metric in metrics_to_analyze:
    ext_rows = []
    for (exp, var), g in long.groupby(["experiment", "variable"]):
        # require enough valid points
        g = g.dropna(subset=[metric, "target_mean"])
        if g.empty:
            continue
        for p in thresholds:
            thr = np.nanpercentile(g["target_mean"].values, p)
            sel = g[g["target_mean"] >= thr]
            ext_rows.append({
                "experiment": exp,
                "variable": var,
                "percentile": p,
                "threshold_value": float(thr),
                "days_count": int(len(sel)),
                "cond_mean": float(sel[metric].mean()) if not sel.empty else np.nan,
            })
    df_ext = pd.DataFrame(ext_rows).sort_values(["variable", "experiment", "percentile"])
    ext_tables[metric] = df_ext
    # save conditional table per metric
    df_ext.to_csv(os.path.join(out_dir, f"metrics_conditional_{metric}_by_percentile.csv"), index=False)
    print(f"Saved conditional table for {metric}")

    # correlations truth vs metric
    corr_rows = []
    for (exp, var), g in long.groupby(["experiment", "variable"]):
        g = g.dropna(subset=[metric, "target_mean"])
        if len(g) < 2:
            corr, pval = np.nan, np.nan
        else:
            try:
                corr, pval = stats.pearsonr(g["target_mean"].values, g[metric].values)
            except Exception:
                corr, pval = np.nan, np.nan
        corr_rows.append({"experiment": exp, "variable": var, "pearson_r": float(corr), "p_value": float(pval), "n": len(g)})
    df_corr = pd.DataFrame(corr_rows).sort_values(["variable", "experiment"])
    corr_tables[metric] = df_corr
    df_corr.to_csv(os.path.join(out_dir, f"metrics_truth_{metric}_correlation.csv"), index=False)
    print(f"Saved correlation table for {metric}")

print("\nSaved conditional & correlation tables for metrics:", ", ".join(metrics_to_analyze))

# Simple plots per variable and per metric: bar of conditional means, and truth vs metric scatter
import matplotlib.cm as cm
vars_to_plot = sorted(long["variable"].dropna().unique())
os.makedirs(os.path.join(out_dir, "plots"), exist_ok=True)

for var in vars_to_plot:
    sub_base = long[long["variable"] == var].dropna(subset=["target_mean"])
    if sub_base.empty:
        continue

    for metric in metrics_to_analyze:
        sub = sub_base.dropna(subset=[metric])
        # Bar: conditional mean for each experiment & percentile -> separate figure
        ext_sub = ext_tables.get(metric, pd.DataFrame())
        ext_sub_var = ext_sub[ext_sub["variable"] == var] if not ext_sub.empty else ext_sub
        fig, ax = plt.subplots(figsize=(4, 4))
        if not ext_sub_var.empty:
            pv = ext_sub_var.pivot(index="experiment", columns="percentile", values="cond_mean")
            pv.plot(kind="bar", ax=ax)
            ax.set_title(f"{var} — Conditional mean {metric} (>=percentiles)")
            ax.set_ylabel(metric.upper())
            ax.grid(True, ls=":", alpha=0.4)
        else:
            ax.text(0.5, 0.5, "No extreme data", ha="center", va="center")
            ax.set_title(f"{var} — Conditional mean {metric} (>=percentiles)")
        plt.tight_layout()
        out_bar = os.path.join(out_dir, "plots", f"cond_{metric}_bar_{var}.pdf")
        fig.savefig(out_bar, bbox_inches="tight", dpi=150)
        print("Saved", out_bar)
        plt.show()
        plt.close(fig)

        # Scatter: truth vs metric per experiment -> separate figure
        exps = sorted(sub["experiment"].unique())
        colors = cm.tab10.colors
        fig, ax = plt.subplots(figsize=(4, 4))
        any_points = False
        for i, exp in enumerate(exps):
            g = sub[sub["experiment"] == exp]
            if g.empty:
                continue
            if "CorrDiff" in exp:
                label="CorrDiff"
            elif "UNet" in exp:
                label="UNET"
            elif "SI" in exp:
                label="CDSI"
            else:
                label=exp
            ax.scatter(g["target_mean"], g[metric], s=10, alpha=0.6, color=colors[i % len(colors)], label=label)
            any_points = True
            # add linear fit line if enough points
            if len(g) >= 2:
                m, b = np.polyfit(g["target_mean"].values, g[metric].values, 1)
                xs = np.linspace(g["target_mean"].min(), g["target_mean"].max(), 50)
                ax.plot(xs, m * xs + b, color=colors[i % len(colors)], linewidth=1)
        if not any_points:
            ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_xlabel("Daily target mean")
        ax.set_ylabel(metric.upper())
        # ax.set_title(f"{var} — truth vs {metric}")
        ax.legend(loc="best", fontsize="large")
        ax.grid(True, ls=":", alpha=0.4)
        plt.tight_layout()
        out_scatter = os.path.join(out_dir, "plots", f"truth_vs_{metric}_scatter_{var}.pdf")
        fig.savefig(out_scatter, bbox_inches="tight", dpi=150)
        print("Saved", out_scatter)
        plt.show()
        plt.close(fig)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os


# Configure names and metric/quantiles to use (edit as needed)
exp_a = experiments[2]           # experiment considered "SI"
exp_b = experiments[1]     # experiment considered "CorrDiff"
metric = "rmse"             # metric to compare: "rmse", "ssr", or "crps"
vars_to_check = sorted(long["variable"].dropna().unique())  # or set specific list
low_q = 0.25                # quantile for "good" (lower is better for rmse)
high_q = 0.75               # quantile for "bad"  (higher is worse for rmse)
out_dir_plots = os.path.join(out_dir, "plots", "compare_SI_CorrDiff")
os.makedirs(out_dir_plots, exist_ok=True)

for var in vars_to_check:
    df_var = long[(long["variable"] == var) & (long["experiment"].isin([exp_a, exp_b]))][["date","experiment", metric]].dropna()
    if df_var.empty:
        print(f"Skipping {var}: no data for {exp_a}/{exp_b} with metric {metric}")
        continue

    piv = df_var.pivot(index="date", columns="experiment", values=metric).dropna()
    if piv.empty or (exp_a not in piv.columns) or (exp_b not in piv.columns):
        print(f"Skipping {var}: not enough overlapping dates for both experiments")
        continue

    # correlation
    try:
        from scipy import stats
        r, p = stats.pearsonr(piv[exp_a].values, piv[exp_b].values)
    except Exception:
        r, p = np.nan, np.nan
    print(f"{var}: Pearson r ({metric}) between {exp_a} and {exp_b} = {r:.3f} (p={p:.3g}), n={len(piv)}")

    # scatter plot with 1:1 line
    fig, ax = plt.subplots(figsize=(6,5))
    ax.scatter(piv[exp_a], piv[exp_b], s=20, alpha=0.7)
    mn = min(piv.min().min(), 0)
    mx = piv.max().max()
    ax.plot([mn,mx],[mn,mx], color="gray", linestyle="--", linewidth=1)
    ax.set_xlabel(f"{exp_a} {metric}")
    ax.set_ylabel(f"{exp_b} {metric}")
    ax.set_title(f"{var}: {exp_a} vs {exp_b} ({metric}), r={r:.2f}")
    ax.grid(True, ls=":", alpha=0.4)
    fn = os.path.join(out_dir_plots, f"scatter_{var}_{metric}_{exp_a}_vs_{exp_b}.pdf")
    fig.savefig(fn, bbox_inches="tight", dpi=150)
    plt.show()
    plt.close(fig)

    # find days where exp_a good & exp_b bad, and reversed
    qa = piv[exp_a].quantile(low_q)
    qb = piv[exp_b].quantile(high_q)
    mask_a_good_b_bad = (piv[exp_a] <= qa) & (piv[exp_b] >= qb)
    df_a_good_b_bad = piv.loc[mask_a_good_b_bad].sort_values(by=[exp_b, exp_a], ascending=[False, True])
    out1 = os.path.join(out_dir_plots, f"{var}_{metric}_{exp_a}_good_{exp_b}_bad.csv")
    df_a_good_b_bad.reset_index().to_csv(out1, index=False)
    print(f"  {var}: {len(df_a_good_b_bad)} days where {exp_a} <= {low_q*100:.0f}p({qa:.3g}) and {exp_b} >= {high_q*100:.0f}p({qb:.3g}) -> saved {out1}")

    qc = piv[exp_b].quantile(low_q)
    qd = piv[exp_a].quantile(high_q)
    mask_b_good_a_bad = (piv[exp_b] <= qc) & (piv[exp_a] >= qd)
    df_b_good_a_bad = piv.loc[mask_b_good_a_bad].sort_values(by=[exp_a, exp_b], ascending=[False, True])
    out2 = os.path.join(out_dir_plots, f"{var}_{metric}_{exp_b}_good_{exp_a}_bad.csv")
    df_b_good_a_bad.reset_index().to_csv(out2, index=False)
    print(f"  {var}: {len(df_b_good_a_bad)} days where {exp_b} <= {low_q*100:.0f}p({qc:.3g}) and {exp_a} >= {high_q*100:.0f}p({qd:.3g}) -> saved {out2}")

    # quick summary prints (first few rows)
    if not df_a_good_b_bad.empty:
        print("  Example days where SI good & CorrDiff bad:")
        print(df_a_good_b_bad.reset_index().head().to_string(index=False))
    if not df_b_good_a_bad.empty:
        print("  Example days where CorrDiff good & SI bad:")
        print(df_b_good_a_bad.reset_index().head().to_string(index=False))

In [ ]:
def worst_days_for_model(long_df: pd.DataFrame,
                         experiment: str,
                         variable: str,
                         metric: str = "rmse",
                         x_percent: float = 10.0,
                         higher_is_worse: bool = True,
                         save_csv: str = None) -> pd.DataFrame:
    """
    Return a DataFrame with the worst x_percent days for given experiment/variable/metric.
    - long_df: long table created earlier (must contain columns date, experiment, variable, metric)
    - x_percent: percentage (0-100) of days to consider worst (e.g. 10 -> top 10% worst days)
    - higher_is_worse: True if larger metric values are worse (rmse, crps), False otherwise
    - save_csv: optional path to save the result
    Returns DataFrame indexed by date with columns: experiment, variable, metric
    """
    sub = long_df[(long_df["experiment"] == experiment) & (long_df["variable"] == variable)].copy()
    if sub.empty:
        raise RuntimeError(f"No data for {experiment}/{variable}")
    sub = sub.dropna(subset=[metric, "date"]).copy()
    sub["date"] = pd.to_datetime(sub["date"])
    if sub.empty:
        raise RuntimeError(f"No valid {metric} values for {experiment}/{variable}")

    pct = float(x_percent)
    if higher_is_worse:
        thr = sub[metric].quantile(1.0 - pct/100.0)
        mask = sub[metric] >= thr
    else:
        thr = sub[metric].quantile(pct/100.0)
        mask = sub[metric] <= thr

    worst = sub.loc[mask, ["date", "experiment", "variable", metric]].sort_values(metric, ascending=higher_is_worse==False)
    worst = worst.set_index("date")
    if save_csv:
        os.makedirs(os.path.dirname(save_csv) or ".", exist_ok=True)
        worst.reset_index().to_csv(save_csv, index=False)
    return worst

def compare_worst_sets(long_df: pd.DataFrame,
                      exp_a: str, exp_b: str,
                      variable: str, metric: str = "rmse",
                      x_percent: float = 10.0,
                      higher_is_worse: bool = True) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Compute worst-x% sets for two experiments and return (only_a, only_b, intersection) as DataFrames.
    Dates are aligned by calendar date strings / datetimes.
    """
    wa = worst_days_for_model(long_df, exp_a, variable, metric, x_percent, higher_is_worse)
    wb = worst_days_for_model(long_df, exp_b, variable, metric, x_percent, higher_is_worse)

    dates_a = set(pd.to_datetime(wa.index).strftime("%Y-%m-%d"))
    dates_b = set(pd.to_datetime(wb.index).strftime("%Y-%m-%d"))

    inter = sorted(dates_a & dates_b)
    only_a = sorted(dates_a - dates_b)
    only_b = sorted(dates_b - dates_a)

    df_inter = pd.concat([wa.loc[wa.index.strftime("%Y-%m-%d").isin(inter)].assign(match="both"),
                          wb.loc[wb.index.strftime("%Y-%m-%d").isin(inter)].assign(match="both")],
                         keys=[exp_a, exp_b], names=["source", "date"], sort=False)
    df_a_only = wa.loc[wa.index.strftime("%Y-%m-%d").isin(only_a)].assign(match=f"{exp_a}_only")
    df_b_only = wb.loc[wb.index.strftime("%Y-%m-%d").isin(only_b)].assign(match=f"{exp_b}_only")

    return df_a_only, df_b_only, df_inter

# # Example usage (after long exists):
# # - get top 10% worst SI days for variable "pr"
# si_worst = worst_days_for_model(long, "test_SI", "pr", metric="rmse", x_percent=10.0, higher_is_worse=True, save_csv="output/si_worst_pr_10p.csv")
# corr_worst = worst_days_for_model(long, "test_CorrDiff", "pr", metric="rmse", x_percent=10.0, higher_is_worse=True, save_csv="output/corr_worst_pr_10p.csv")
# only_si, only_corr, both = compare_worst_sets(long, "test_SI", "test_CorrDiff", "pr", metric="rmse", x_percent=10.0)
# print("SI-only worst days:", len(only_si), "CorrDiff-only:", len(only_corr), "Both:", len(both))

# # Quick plot: mark worst days on time series for SI and CorrDiff
# var = "pr"
# x_percent = 10.0
# si_w = worst_days_for_model(long, "test_SI", var, "rmse", x_percent)
# corr_w = worst_days_for_model(long, "test_CorrDiff", var, "rmse", x_percent)
# piv = long[long["variable"]==var].pivot(index="date", columns="experiment", values="rmse")
# piv.index = pd.to_datetime(piv.index)
# fig, ax = plt.subplots(figsize=(10,4))
# # ax.plot(piv.index, piv["test_SI"], label="SI", marker="o")
# # ax.plot(piv.index, piv["test_CorrDiff"], label="CorrDiff", marker="o")
# ax.scatter(si_w.index, si_w["rmse"], color="C0", s=60, edgecolor="k", label=f"SI worst {x_percent}%")
# ax.scatter(corr_w.index, corr_w["rmse"], color="C1", s=60, edgecolor="k", label=f"CorrDiff worst {x_percent}%")
# ax.legend(loc="best"); ax.set_title(f"{var} RMSE with worst {x_percent}% highlighted"); plt.show()

In [ ]:
from typing import Union, List, Sequence

def get_worst_dates(long_df: pd.DataFrame,
                    experiment: Union[str, Sequence[str]],
                    variable: str,
                    metric: str = "rmse",
                    x_percent: float = 10.0,
                    higher_is_worse: bool = True,
                    combine: str = "union") -> pd.DatetimeIndex:
    """
    Return DatetimeIndex of the worst x_percent days.

    - experiment: single experiment name or iterable of names.
    - combine: how to combine multiple reference experiments:
        * "union" (default)  - union of each experiment's worst-x% dates,
        * "intersection"      - intersection of each experiment's worst-x% dates,
        * "pooled"            - compute threshold from pooled metric values across refs, then take matching dates,
        * "per_ref"           - return concatenated unique dates but kept per-ref behaviour is handled by callers
    Returns a DatetimeIndex (sorted, unique). For "per_ref" mode this returns the union of per-ref dates
    (use compare_performance_on_worst_dates with combine='per_ref' to get per-ref comparisons).
    """
    refs = [experiment] if isinstance(experiment, str) else list(experiment)

    if len(refs) == 0:
        return pd.DatetimeIndex([])

    # simple single-experiment path
    if len(refs) == 1:
        worst = worst_days_for_model(long_df, refs[0], variable, metric=metric,
                                     x_percent=x_percent, higher_is_worse=higher_is_worse)
        dates = pd.to_datetime(worst.index).unique()
        return pd.DatetimeIndex(sorted(dates))

    # multiple refs
    combine = combine.lower()
    if combine == "pooled":
        sub = long_df[(long_df["experiment"].isin(refs)) & (long_df["variable"] == variable)].dropna(subset=[metric, "date"]).copy()
        if sub.empty:
            return pd.DatetimeIndex([])
        sub["date"] = pd.to_datetime(sub["date"])
        pct = float(x_percent)
        if higher_is_worse:
            thr = sub[metric].quantile(1.0 - pct/100.0)
            mask = sub[metric] >= thr
        else:
            thr = sub[metric].quantile(pct/100.0)
            mask = sub[metric] <= thr
        dates = pd.to_datetime(sub.loc[mask, "date"].unique())
        return pd.DatetimeIndex(sorted(dates))

    # union / intersection: compute per-ref worst dates then combine
    per_ref_dates = []
    for r in refs:
        w = worst_days_for_model(long_df, r, variable, metric=metric,
                                 x_percent=x_percent, higher_is_worse=higher_is_worse)
        per_ref_dates.append(set(pd.to_datetime(w.index).strftime("%Y-%m-%d")))

    if combine == "intersection":
        ds = sorted(set.intersection(*per_ref_dates)) if per_ref_dates else []
    else:  # union (default) or unknown -> union
        ds = sorted(set.union(*per_ref_dates))

    dates = pd.DatetimeIndex(pd.to_datetime(ds))
    return dates

def compare_performance_on_worst_dates(long_df: pd.DataFrame,
                                       ref_exp: Union[str, Sequence[str]],
                                       other_exp: Union[str, Sequence[str]],
                                       variable: str,
                                       metric: str = "rmse",
                                       x_percent: float = 10.0,
                                       higher_is_worse: bool = True,
                                       combine: str = "union",
                                       save_csv: str = None,
                                       show_plot: bool = False):
    """
    For the worst x_percent dates of ref_exp (single or multiple), return a table with metric
    for the ref(s) and the other_exp(s) and a small summary.

    - ref_exp: single name or list of reference experiments.
    - other_exp: single name or list of experiments to compare.
    - combine: how to treat multiple refs:
        * "union" / "intersection" / "pooled" -> compute one combined date set and return (df, summary)
        * "per_ref" -> return a dict mapping each ref -> (df, summary)
    Returns:
      - if combine == "per_ref": dict { ref_name: (df_by_date, summary_series) }
      - otherwise: (df_by_date, summary_series)
    """
    refs = [ref_exp] if isinstance(ref_exp, str) else list(ref_exp)
    others = [other_exp] if isinstance(other_exp, str) else list(other_exp)

    if combine == "per_ref" and len(refs) > 1:
        # produce per-ref comparisons
        out = {}
        for r in refs:
            df, summary = compare_performance_on_worst_dates(
                long_df, r, others, variable, metric=metric, x_percent=x_percent,
                higher_is_worse=higher_is_worse, combine="union", save_csv=None, show_plot=show_plot
            )
            out[r] = (df, summary)
            # optionally save each
            if save_csv:
                base, ext = os.path.splitext(save_csv)
                fn = f"{base}_{r}{ext or '.csv'}"
                os.makedirs(os.path.dirname(fn) or ".", exist_ok=True)
                df.reset_index().to_csv(fn, index=False)
        return out

    # compute combined dates (combine param handled by get_worst_dates)
    dates = get_worst_dates(long_df, refs, variable, metric=metric, x_percent=x_percent,
                            higher_is_worse=higher_is_worse, combine=combine)
    if len(dates) == 0:
        raise RuntimeError("No worst dates found")

    # ensure long_df.date is datetime
    tmp = long_df.copy()
    tmp["date"] = pd.to_datetime(tmp["date"], errors="coerce")

    # prepare result index
    df = pd.DataFrame(index=dates)
    df.index.name = "date"

    # fill ref columns: for multiple refs create one column per ref (use mean if multiple refs requested for other side)
    for r in refs:
        s = tmp[(tmp["experiment"] == r) & (tmp["variable"] == variable)].set_index("date")[metric]
        df[r] = s.reindex(dates).values

    # fill other columns
    for o in others:
        s = tmp[(tmp["experiment"] == o) & (tmp["variable"] == variable)].set_index("date")[metric]
        df[o] = s.reindex(dates).values

    # If single reference and single other, keep old-style summary; otherwise build summary per column
    summary_dict = {}
    for col in df.columns:
        summary_dict[f"{col}_mean"] = float(df[col].mean(skipna=True))
        summary_dict[f"{col}_median"] = float(df[col].median(skipna=True))
        summary_dict[f"{col}_count"] = int(df[col].count())

    # add global info
    summary_dict["n_worst_dates"] = int(len(dates))
    # if exactly one ref and one other, keep a mean_diff key for compatibility
    if len(refs) == 1 and len(others) == 1:
        summary_dict["mean_diff_ref_minus_other"] = float((df[refs[0]] - df[others[0]]).mean(skipna=True))
    else:
        # add pairwise differences (ref - other) averages
        for r in refs:
            for o in others:
                summary_dict[f"mean_diff_{r}_minus_{o}"] = float((df[r] - df[o]).mean(skipna=True))

    summary = pd.Series(summary_dict)

    if save_csv and not isinstance(save_csv, bool):
        os.makedirs(os.path.dirname(save_csv) or ".", exist_ok=True)
        df_reset = df.reset_index()
        df_reset.to_csv(save_csv, index=False)

    if show_plot:
        try:
            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(figsize=(8, max(3, len(df)/8)))
            for col in df.columns:
                ax.plot(df.index, df[col], marker="o", linestyle="-", label=col)
            ax.set_xlabel("Date")
            ax.set_ylabel(metric)
            title_refs = ",".join(refs)
            title_others = ",".join(others)
            ax.set_title(f"Worst {x_percent}% dates for {title_refs} — compare {title_others}")
            ax.legend(loc="best", fontsize="small")
            ax.grid(True, ls=":", alpha=0.6)
            fig.autofmt_xdate()
            plt.show()
        except Exception:
            pass

    return df, summary

# Example usage:
df_cmp_multi, stats_multi = compare_performance_on_worst_dates(long, experiments[2], [experiments[1], experiments[0]], "tas",
                                                               metric="rmse", x_percent=10.0,
                                                               combine="union", save_csv=f"{output_dir}/worst_SI_vs_others_tas.csv",
                                                               show_plot=False)
print(stats_multi)


ref_col = experiments[2]  # set to the reference experiment name you used above
if ref_col in df_cmp_multi.columns:
    df_cmp_multi = df_cmp_multi.sort_values(by=ref_col, ascending=False, na_position="last")
else:
    # fallback: sort by first numeric column if reference column not present
    num_cols = df_cmp_multi.select_dtypes(include="number").columns.tolist()
    if num_cols:
        df_cmp_multi = df_cmp_multi.sort_values(by=num_cols[0], ascending=False, na_position="last")

print(stats_multi)
print(df_cmp_multi.head())

In [ ]:
import numpy as np
import pandas as pd

# produce LaTeX table for UNET worst-x% days (robust selection using nlargest)
ref = experiments[0]  # UNET experiment name
others = [experiments[1], experiments[2]]  # CorrDiff, SI experiment names
vars_to_do = ["pr", "tas"]
x_percent = -5.0
exp_label = {ref: "UNET", others[0]: "CorrDiff", others[1]: "SI"}  # map names to labels used in table

tmp = long.copy()
tmp["date"] = pd.to_datetime(tmp["date"]).dt.normalize()

def worst_dates_nlargest(long_df, experiment, variable, metric="rmse", x_percent=10.0):
    """
    Select extreme dates for an experiment/variable:
      - positive x_percent -> take top x% (worst = largest metric)
      - negative x_percent -> take bottom |x_percent|% (best = smallest metric)
    Returns DatetimeIndex of selected dates (sorted).
    """
    sub = long_df[(long_df["experiment"] == experiment) & (long_df["variable"] == variable)].dropna(subset=[metric, "date"]).copy()
    sub["date"] = pd.to_datetime(sub["date"]).dt.normalize()
    N = len(sub)
    if N == 0:
        return pd.DatetimeIndex([])
    pct = float(x_percent)
    k = int(np.ceil(N * abs(pct) / 100.0))
    k = max(1, min(k, N))
    if pct < 0:
        # best days -> smallest metric values
        sel = sub.nsmallest(k, metric)
    else:
        # worst days -> largest metric values
        sel = sub.nlargest(k, metric)
    return pd.DatetimeIndex(sorted(pd.to_datetime(sel["date"].values)))

def fmt_val(x):
    if pd.isna(x):
        return "{--}"
    return f"{x:.3f}"

lines = []
lines.append(r"\begin{tabular*}{\linewidth}{@{\extracolsep{\fill}} l S S S @{} }")
lines.append(r"  \toprule")
lines.append(r"  Model & {RMSE} & {SSR} & {CRPS} \\")
lines.append(r"  \midrule")
lines.append("")

# loop variables
for var in vars_to_do:
    if var == "pr":
        lines.append(r"  \multicolumn{4}{l}{\textbf{Precipitation}} \\")
    else:
        lines.append("")
        lines.append(r"  \addlinespace[0.6em]")
        lines.append(r"  \multicolumn{4}{l}{\textbf{Temperature (2\,m)}} \\")
    lines.append(r"  \addlinespace[0.3em]")

    # get UNET worst dates (exact k worst days)
    dates = worst_dates_nlargest(tmp, ref, var, metric="rmse", x_percent=x_percent)
    # compute metrics for each experiment on those dates
    for exp in [ref, others[0], others[1]]:  # order: UNET, CorrDiff, SI
        sel = tmp[(tmp["experiment"] == exp) & (tmp["variable"] == var) & (tmp["date"].isin(dates))]
        mean_rmse = sel["rmse"].mean(skipna=True) if not sel.empty else np.nan
        mean_ssr = sel["ssr"].mean(skipna=True) if ("ssr" in sel.columns and not sel.empty) else np.nan
        mean_crps = sel["crps"].mean(skipna=True) if ("crps" in sel.columns and not sel.empty) else np.nan
        label = exp_label.get(exp, exp)
        lines.append(f"  {label} & {fmt_val(mean_rmse)} & {fmt_val(mean_ssr)} & {fmt_val(mean_crps)} \\\\")
    lines.append("")

lines.append(r"  \bottomrule")
lines.append(r"\end{tabular*}")

latex_table = "\n".join(lines)
print(latex_table)

In [ ]:
# ...existing code...
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# ...existing code...
def plot_metrics_vs_unet(long_df: pd.DataFrame,
                         ref: str = "test_UNet_small_r2",
                         others: list = ["test_SI_small_r2", "test_CorrDiff_small_r2"],
                         variables: list = None,
                         metrics: list = ["rmse", "ssr", "crps"],
                         out_dir: str = "output/plots/metrics_vs_unet"):
    """
    Scatter plots of each metric (y) for others vs UNET RMSE (x).
    Produces one figure per variable with one subplot per metric.
    """
    os.makedirs(out_dir, exist_ok=True)
    df = long_df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    if variables is None:
        variables = sorted(df["variable"].dropna().unique())

    for var in variables:
        sub = df[df["variable"] == var].copy()
        if sub.empty:
            print(f"Skipping variable {var}: no data")
            continue

        fig, axes = plt.subplots(1, len(metrics), figsize=(5 * len(metrics), 5), sharey=False)
        if len(metrics) == 1:
            axes = [axes]

        for ax, metric in zip(axes, metrics):
            # pivot RMSE separately (always needed for x-axis) and pivot the requested metric for y
            piv_rmse = sub.pivot(index="date", columns="experiment", values="rmse")
            piv_metric = sub.pivot(index="date", columns="experiment", values=metric)

            # if there's no RMSE for any experiment, nothing to plot
            if piv_rmse.dropna(how="all").empty:
                ax.text(0.5, 0.5, "No RMSE data available", ha="center", va="center")
                ax.set_xlabel("UNET RMSE")
                ax.set_ylabel(metric.upper())
                ax.set_title(f"{var} — {metric} vs UNET RMSE")
                continue

            if ref not in piv_rmse.columns:
                ax.text(0.5, 0.5, f"Reference '{ref}' missing RMSE", ha="center", va="center")
                print(f"Variable {var}, metric {metric}: reference '{ref}' has no RMSE -> skipping subplot")
                ax.set_xlabel("UNET RMSE")
                ax.set_ylabel(metric.upper())
                ax.set_title(f"{var} — {metric} vs UNET RMSE")
                continue

            x_all = piv_rmse[ref].dropna()
            any_points = False

            for other in others:
                # other must have the metric (y). If it doesn't, skip.
                if other not in piv_metric.columns:
                    print(f"  Variable {var}, metric {metric}: other '{other}' has no '{metric}' -> skipping")
                    continue

                data = pd.concat([x_all, piv_metric[other]], axis=1).dropna()
                if data.empty:
                    # no overlapping dates between ref RMSE and this other's metric
                    continue
                if "CorrDiff" in other:
                    label="CorrDiff"
                elif "UNet" in other:
                    label="UNET"
                elif "SI" in other:
                    label="CDSI"
                else:
                    label=other
                any_points = True
                x = data.iloc[:, 0].values
                y = data.iloc[:, 1].values
                ax.scatter(x, y, s=20, alpha=0.7, label=label)

                try:
                    slope, intercept = np.polyfit(x, y, 1)
                    xs = np.linspace(x.min(), x.max(), 50)
                    ax.plot(xs, slope * xs + intercept, linestyle="--", linewidth=1, alpha=0.8)
                except Exception:
                    pass

                # try:
                #     r, p = stats.pearsonr(x, y)
                #     ax.annotate(f"{other}: r={r:.2f}", xy=(0.05, 0.78 - 0.06 * others.index(other)),
                #                 xycoords="axes fraction", fontsize=9)
                # except Exception:
                #     pass

            if not any_points:
                ax.text(0.5, 0.5, "No overlapping points", ha="center", va="center")
            ax.set_xlabel("UNET RMSE")
            ax.set_ylabel(metric.upper())
            var_name = "Precipitation" if var == "pr" else "Temperature (2m)" if var == "tas" else var
            ax.set_title(f"{var_name} — {metric.upper()}")
            ax.grid(True, ls=":", alpha=0.4)
            if any_points:
                ax.legend(fontsize="large")
            # 1:1 line only makes sense for rmse vs rmse
            if metric == "rmse":
                try:
                    mn = min(piv_rmse.min().min(), piv_metric.min().min())
                    mx = max(piv_rmse.max().max(), piv_metric.max().max())
                    if np.isfinite(mn) and np.isfinite(mx):
                        ax.plot([mn, mx], [mn, mx], color="gray", linestyle=":", linewidth=1)
                except Exception:
                    pass

        plt.tight_layout()
        out_png = os.path.join(out_dir, f"metrics_vs_unet_{var}.pdf")
        fig.savefig(out_png, dpi=150, bbox_inches="tight")
        print(f"Saved {out_png}")
        plt.show()
        plt.close(fig)
# ...existing code...

# Example usage (run this cell after 'long' exists):
# quick diagnostics
print("experiments:", experiments)
print("experiments present in long:", sorted(long["experiment"].unique()))

# corrected call: UNET as ref, SI and CorrDiff as others
plot_metrics_vs_unet(
    long,
    ref=experiments[2],                 # UNET
    others=[experiments[0], experiments[1]],  # SI, CorrDiff
    variables=["pr", "tas"],
    metrics=["rmse", "ssr", "crps"],
    out_dir=f"{output_dir}/plots/metrics_vs_unet",
)


In [ ]:
import pandas as pd
print("Columns:", long.columns.tolist())
print(long.groupby(["experiment","variable"])[["rmse","ssr","crps"]].apply(lambda g: g.notna().sum()))
# per-experiment counts and NaNs
print("\nNaN fractions:")
print(long.groupby(["experiment","variable"])[["ssr","crps"]].apply(lambda g: g.isna().mean()).unstack(level=0))
# show a few rows where crps or ssr exist
print("\nExample rows with CRPS or SSR present:")
print(long[long["crps"].notna()].head())
print(long[long["ssr"].notna()].head())

In [ ]:
# Sanity checking the numbers

import numpy as np

def destandardize(
    sample,
    pr_stats_path='/mimer/NOBACKUP/groups/mlhighres/projects/detex/HCLIM_EC-Earth3-Veg/standardized_new/mean_std_remapped.pr_EUR-11_EC-Earth3-Veg_historical_r1i1p1f1_HCLIMcom-SMHI_HCLIM43-ALADIN_v1-r1_day_1951-2014_mm_day_noleap.npy', 
    tas_stats_path='/mimer/NOBACKUP/groups/mlhighres/projects/detex/HCLIM_EC-Earth3-Veg/standardized_new/mean_std_remapped.tas_EUR-11_EC-Earth3-Veg_historical_r1i1p1f1_HCLIMcom-SMHI_HCLIM43-ALADIN_v1-r1_day_1951-2014_noleap.npy',
    std_dataset=False
):
    pr_mean, pr_std = np.load(pr_stats_path)
    tas_mean, tas_std = np.load(tas_stats_path)

    if std_dataset:
        return sample * np.array([pr_std, tas_std])
    else:
        return sample * np.array([pr_std, tas_std]) + np.array([pr_mean, tas_mean])
    
pr_mean, pr_std = np.load('/mimer/NOBACKUP/groups/mlhighres/projects/detex/HCLIM_EC-Earth3-Veg/standardized_new/mean_std_remapped.pr_EUR-11_EC-Earth3-Veg_historical_r1i1p1f1_HCLIMcom-SMHI_HCLIM43-ALADIN_v1-r1_day_1951-2014_mm_day_noleap.npy')
tas_mean, tas_std = np.load('/mimer/NOBACKUP/groups/mlhighres/projects/detex/HCLIM_EC-Earth3-Veg/standardized_new/mean_std_remapped.tas_EUR-11_EC-Earth3-Veg_historical_r1i1p1f1_HCLIMcom-SMHI_HCLIM43-ALADIN_v1-r1_day_1951-2014_noleap.npy')

print(f"pr mean: {pr_mean}, pr std: {pr_std}")
print(f"tas mean: {tas_mean}, tas std: {tas_std}")


In [ ]:
# ...existing code...
def generate_scaled_latex(pr_std: float, tas_std: float) -> str:
    """
    Scale RMSE and CRPS in the hard-coded table by pr_std / tas_std and
    return a LaTeX table fragment as a single string.
    """
    # table data: (Model, RMSE, SSR, CRPS) -- keep SSR as-is
    pr_rows = [
        ("UNET XS",        0.747, "{--}", "{--}"),
        ("EDM XS 5 ens",   0.937, 0.952, 0.329),
        ("CorrDiff XS 5 ens", 0.820, 0.909, 0.269),
        ("CorrDiff XS 20 ens",0.779, 0.894, 0.269),
        ("CorrDiff XS 50 ens",0.768, 0.891, 0.269),
        ("CorrDiff XS 100 ens",0.765, 0.892, 0.269),
        ("SI XS 40 steps 5 ens", 0.797, 0.715, 0.272),
        ("SI XS 40 steps 20 ens",0.770, 0.690, 0.272),
        ("SI XS 40 steps 50 ens",0.764, 0.685, 0.272),
        ("SI XS 40 steps 100 ens",0.762, 0.684, 0.271),
    ]

    tas_rows = [
        ("UNET XS",        0.202, "{--}", "{--}"),
        ("EDM XS 5 ens",   0.318, 0.724, 0.157),
        ("CorrDiff XS 5 ens", 0.234, 0.638, 0.108),
        ("CorrDiff XS 20 ens",0.228, 0.614, 0.108),
        ("CorrDiff XS 50 ens",0.227, 0.607, 0.108),
        ("CorrDiff XS 100 ens",0.227, 0.605, 0.108),
        ("SI XS 40 steps 5 ens", 0.220, 0.629, 0.097),
        ("SI XS 40 steps 20 ens",0.213, 0.606, 0.096),
        ("SI XS 40 steps 50 ens",0.214, 0.597, 0.097),
        ("SI XS 40 steps 100 ens",0.213, 0.597, 0.097),
    ]

    def fmt(val):
        return f"{val:.3f}"

    lines = []
    lines.append(r"Model & {RMSE} & {SSR} & {CRPS} \\")
    lines.append(r"\midrule")
    lines.append("")
    lines.append(r"\multicolumn{4}{l}{\textbf{Precipitation}} \\")
    lines.append(r"\addlinespace[0.3em]")
    for m, rmse, ssr, crps in pr_rows:
        # scale RMSE and CRPS by pr_std if numeric
        rmse_s = fmt(rmse * pr_std) if isinstance(rmse, (int,float)) else rmse
        crps_s = fmt(crps * pr_std) if isinstance(crps, (int,float)) else crps
        ssr_s = f"{ssr:.3f}" if isinstance(ssr, (int,float)) else ssr
        lines.append(f"{m} & {rmse_s} & {ssr_s} & {crps_s} \\\\")
    lines.append("")
    lines.append(r"\addlinespace[0.6em]")
    lines.append("")
    lines.append(r"\multicolumn{4}{l}{\textbf{Temperature (2\,m)}} \\")
    lines.append(r"\addlinespace[0.3em]")
    for m, rmse, ssr, crps in tas_rows:
        rmse_s = fmt(rmse * tas_std) if isinstance(rmse, (int,float)) else rmse
        crps_s = fmt(crps * tas_std) if isinstance(crps, (int,float)) else crps
        ssr_s = f"{ssr:.3f}" if isinstance(ssr, (int,float)) else ssr
        lines.append(f"{m} & {rmse_s} & {ssr_s} & {crps_s} \\\\")
    # join into single latex string
    return "\n".join(lines)

print(generate_scaled_latex(pr_std, tas_std))


In [ ]:
0.937 * pr_std, 0.329 * pr_std

In [ ]:
0.220 * tas_std, 0.097 * tas_std